# Loading

In [ ]:
# Set environment variables FIRST (lightweight)
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ['MUJOCO_GL'] = 'egl'
os.environ['PYOPENGL_PLATFORM'] = 'egl'
os.environ["XLA_FLAGS"] = "--xla_gpu_triton_gemm_any=True"
os.environ["JAX_CAPTURED_CONSTANTS_REPORT_FRAMES"]="-1"

In [ ]:
import mujoco
import numpy as np
import mediapy as media
import ipywidgets as widgets
from IPython.display import display, clear_output

In [ ]:
# JAX setup (this is the slow import - takes ~60s on HPC)
from pathlib import Path
project_root = Path.cwd().parents[0]
import jax
jax.config.update("jax_compilation_cache_dir", (project_root / "tmp/jax_cache").as_posix())
jax.config.update("jax_persistent_cache_min_entry_size_bytes", -1)
jax.config.update("jax_persistent_cache_min_compile_time_secs", 0)
try:
    jax.config.update("jax_persistent_cache_enable_xla_caches", "xla_gpu_per_fusion_autotune_cache_dir")
except AttributeError:
    pass
jax.config.update("jax_default_matmul_precision", "high")
num_gpus = jax.device_count(backend="gpu")
print(f"JAX initialized with {num_gpus} GPU(s)")

In [ ]:
from fly_mimic.utils.rollout_saver import Rollout, HDF5RolloutSaver
from fly_mimic.utils.data_utils import ReferenceClips
from natsort import natsorted
from omegaconf import OmegaConf
from fly_neuromechanics.utils.path_utils import (
    register_custom_resolvers, load_config_and_override_paths, convert_dict_to_path
)
from fly_neuromechanics.vnc_utils import (
    load_wTable, load_W, create_vnc_params
)

from pathlib import Path
from mujoco_playground._src.mjx_env import get_qpos_ids
import jax.numpy as jnp
from functools import partial
from fly_mimic.utils.fly_logging import add_arrows
from fly_mimic.utils.utils import add_text_labels, add_cross_hair_sites, add_trajectory_sites_spheres
from fly_mimic.envs.fruitfly.constants import _WING_PARAMS
from fly_mimic.envs.fruitfly.closed_loop_imitation import ClosedLoopImitation

In [ ]:
data_type = 'V2_stationary'
project_dir = Path.cwd().parent
data_dir = Path('/gscratch/portia/eabe/fly_neuromech')
# data_dir = Path('/gpfs/projects/portia/eabe/fly_neuromech')

In [ ]:
version = 'walk'
base_dir = Path(f'/gscratch/portia/eabe/fly_neuromech/{version}')
# base_dir = Path(f'/data2/users/eabe/fly_neuromech/{version}')
run_cfg_list = natsorted(list(Path(base_dir).rglob('run_config*.yaml')))
for n, run_cfg in enumerate(run_cfg_list):
    temp = OmegaConf.load(run_cfg)
    print(f'{n}:', temp.version, run_cfg)

# ###### Load and update config with specified paths template ###### 
cfg_num = -3

# NEW APPROACH: Load config a"nd replace paths using workstation.yaml template
cfg = load_config_and_override_paths(
    config_path=run_cfg_list[cfg_num],
    new_paths_template="hyak",    # Use workstation.yaml for local paths
    config_dir=Path.cwd().parent / "configs",
    verbose=False
)

print(f'✅ Loaded experiment: {cfg_num}, {cfg.version}: {run_cfg_list[cfg_num]}')

# Convert string paths to Path objects and create directories
cfg.paths = convert_dict_to_path(cfg.paths)
print("✅ Successfully converted all paths to Path objects and created directories")

fig_dir = cfg.paths.fig_dir
# media.set_show_save_dir(fig_dir)

reference_path = cfg.paths.data_dir/ f"datasets/{cfg.dataset.clip_idx}"
# reference_path = Path('/data2/users/eabe/fly_neuromech/data/datasets/Fruitfly_v2_muscles_stationary_1000hz_interp_padded2.h5')

# reference_path = cfg.paths.data_dir/ f"clips/{cfg.dataset.clip_idx}"
# reference_clips = ReferenceClips.from_path(reference_path, enable_jax=True)
print(f"✅ Loading reference data from {reference_path}")
reference_clips = ReferenceClips.from_path(reference_path, enable_jax=True)

env = ClosedLoopImitation(cfg, reference_clips)
 

wTable = load_wTable(cfg.vnc_sim.wPath/cfg.vnc_sim.dfPath)


In [ ]:
if data_type == 'V1_walking':
    policy_file = data_dir/ 'walk/33721394/policy_data/rollout.h5'
    XML_PATH       = project_dir / 'fruitfly_body_models/fruitfly_v1/fruitfly_v1_free.xml'
    REF_CLIPS_PATH = data_dir / 'data/datasets/Fruitfly_v1_walk_1000hz_interp_padded2.h5'
elif data_type == 'V2_walking':
    XML_PATH       = project_dir / 'fruitfly_body_models/fruitfly_v2.1/fruitfly_v2.1_muscles_viz.xml'
    REF_CLIPS_PATH = data_dir / 'data/datasets/Fruitfly_v2_muscles_free_walking_1000hz_interp_padded2.h5'
elif data_type == 'V1_flight':
    XML_PATH       = project_dir / 'fruitfly_body_models/fruitfly_v1/fruitfly_v1.xml'
    REF_CLIPS_PATH = '/gscratch/portia/eabe/biomech_model/Flybody/datasets/flight/clips/FruitflyV1_flight_kinematics_1000window_10000hz.h5'
elif data_type == 'V2_stationary':
    XML_PATH       = project_dir / 'fruitfly_body_models/fruitfly_v2.1/fruitfly_v2.1_muscles.xml'
    REF_CLIPS_PATH = data_dir / 'data/datasets/Fruitfly_v2_muscles_stationary_1000hz_interp_padded2.h5'
    policy_file = data_dir/ 'walk/33833601/policy_data/rollout.h5'
    
    # REF_CLIPS_PATH = '/data/users/eabe/biomech_model/Flybody/datasets/flight/clips/FruitflyV1_flight_kinematics_1000window_10000hz.h5'
    # REF_CLIPS_PATH = data_dir / 'Flybody/datasets/flight/clips/FruitflyV1_flight_kinematics_1000window_10000hz.h5'
    
# policy_file = Path('/data2/users/eabe/fly_neuromech/walk/33676654/policy_data/rollout.h5')
# policy_file = Path('/data2/users/eabe/fly_neuromech/walk/33656783/policy_data/rollout2.h5')
# policy_file = Path('/data/users/eabe/biomech_model/Flybody/fly_imitation/flight/run_id=31647525/policy_data/policy_data_005060689920_quasi.h5')
saver = HDF5RolloutSaver(policy_file)

FLOOR_XML_PATH = project_dir / 'fruitfly_body_models/fruitfly_v2.1/floor.xml'

# Use MjSpec so <include> directives in the fly XML resolve relative to the
# file's own directory (from_xml_string lacks that context).
fly_spec   = mujoco.MjSpec.from_file(XML_PATH.as_posix())
floor_spec = mujoco.MjSpec.from_file(FLOOR_XML_PATH.as_posix())
spawn_frame = floor_spec.worldbody.add_frame(
                pos=[0,0,-0.005],
                quat=[1,0,0,0],
            )
# Attach floor's worldbody (plane geom + grid material/texture) into fly worldbody.
spawn_body = spawn_frame.attach_body(fly_spec.body("thorax"), "", "")
ref_clips = ReferenceClips.from_path(REF_CLIPS_PATH)
clip_idx = 1
clip_length = ref_clips.clip_lengths[clip_idx]
# # floor_spec = add_cross_hair_sites(floor_spec, init_pos=ref_clips.qpos[clip_idx,0,:3])
# floor_spec = add_trajectory_sites_spheres(spec=floor_spec,
#                                 n_traj_sites=ref_clips.clip_lengths[clip_idx],
#                                 traj=ref_clips.qpos[clip_idx,:,:3])
mj_model = floor_spec.compile()
mj_data  = mujoco.MjData(mj_model)
single_clip = saver.load_rollout(clip_idx, clip_length=clip_length)
abd_idxs = jnp.asarray([get_qpos_ids(mj_model, [joint.name for joint in floor_spec.joints if ('abdomen' in joint.name)])])
wing_joint_idxs = jnp.asarray([get_qpos_ids(mj_model, [joint.name for joint in floor_spec.joints if ('wing' in joint.name)])])
if data_type == 'V1_flight':
    # For flight, set wings to mid-flap so we can see them in the pose tuner.
    init_qpos = jnp.zeros(mj_model.nq)
    init_qpos = init_qpos.at[wing_joint_idxs].set(_WING_PARAMS['default_qpos'])
else:
    init_qpos = jnp.zeros(mj_model.nq)
    init_qpos = init_qpos.at[wing_joint_idxs].set(0.0)
    init_qpos = init_qpos.at[abd_idxs].set(0.0)
mj_data.qpos = init_qpos
mujoco.mj_forward(mj_model, mj_data)
print(f'nq={mj_model.nq}, njnt={mj_model.njnt}, ngeom={mj_model.ngeom}')


In [ ]:
# Build joint index map for all leg joints
LEG_SEGMENTS = ['coxa', 'trochanter', 'femur', 'tibia', 'tarsus']

leg_joints = {}  # name -> qposadr
for jnt_id in range(mj_model.njnt):
    name = mujoco.mj_id2name(mj_model, mujoco.mjtObj.mjOBJ_JOINT, jnt_id)
    if name and any(seg in name for seg in LEG_SEGMENTS):
        leg_joints[name] = int(mj_model.jnt_qposadr[jnt_id])

print(f'Found {len(leg_joints)} leg joints')
for name, idx in leg_joints.items():
    print(f'  [{idx:3d}] {name:40s}  springref={mj_model.qpos_spring[idx]:.4f} rad')

In [ ]:
# ── Geom categorization by body segment ──────────────────────────────────────
orig_geom_rgba = mj_model.geom_rgba.copy()  # saved for color reset
orig_mat_rgba  = mj_model.mat_rgba.copy()   # saved for material color reset

BODY_CATEGORIES = [
    'thorax', 'head', 'abdomen',
    'eye_red',
    'antenna_left', 'antenna_right',
    'haltere_left', 'haltere_right',
    'proboscis',
    'wing_left', 'wing_right',
    'T1_left', 'T1_right',
    'T2_left', 'T2_right',
    'T3_left', 'T3_right',
]

def _body_to_cat(name):
    if not name:
        return None
    n = name.lower()
    # Named appendages checked BEFORE T1/T2/T3 to avoid false segment matches
    if 'antenna' in n:
        return f'antenna_{"left" if "left" in n else "right"}'
    if 'haltere' in n:
        return f'haltere_{"left" if "left" in n else "right"}'
    # Proboscis parts: rostrum, haustellum, labrum
    if any(k in n for k in ('rostrum', 'haustellum', 'labrum')):
        return 'proboscis'
    for seg in ['t1', 't2', 't3']:
        if seg in n:
            return f'{seg.upper()}_{"left" if "left" in n else "right"}'
    if 'wing' in n:
        return f'wing_{"left" if "left" in n else "right"}'
    # Eyes: geom names are head_red / head_black
    if 'red' in n:
        return 'eye_red'
    if 'black' in n:
        return 'eye_black'
    if 'eye' in n:
        return f'eye_{"red" if "left" in n else "black"}'
    for top in ['head', 'thorax', 'abdomen']:
        if top in n:
            return top
    return None

geom_categories = {cat: [] for cat in BODY_CATEGORIES}
for geom_id in range(mj_model.ngeom):
    body_id   = mj_model.geom_bodyid[geom_id]
    body_name = mujoco.mj_id2name(mj_model, mujoco.mjtObj.mjOBJ_BODY, body_id) or ''
    geom_name = mujoco.mj_id2name(mj_model, mujoco.mjtObj.mjOBJ_GEOM, geom_id) or ''
    # head_red / head_black are geom names inside the 'head' body.
    # The body name is just 'head', so we must check the geom name first.
    if 'head' in body_name.lower():
        cat = _body_to_cat(geom_name) or _body_to_cat(body_name)
    else:
        cat = _body_to_cat(body_name)
    if cat in geom_categories:
        geom_categories[cat].append(geom_id)

# ── Material tracking ─────────────────────────────────────────────────────────
# Geoms with material="red"/"black" (like compound eyes) are rendered via
# mat_rgba, not geom_rgba.  Find materials exclusively owned by one category.
_tmp_mat = {}
for cat, idxs in geom_categories.items():
    for i in idxs:
        mid = int(mj_model.geom_matid[i])
        if mid >= 0:
            _tmp_mat.setdefault(mid, set()).add(cat)
_cat_exclusive_mat_ids = {
    mid: next(iter(cats))
    for mid, cats in _tmp_mat.items() if len(cats) == 1
}
del _tmp_mat
print(f'Exclusive-material categories: {sorted(set(_cat_exclusive_mat_ids.values()))} '
      f'({len(_cat_exclusive_mat_ids)} material(s))')

def _cat_default_hex(cat):
    """Representative hex color for a category (checks mat_rgba for material geoms)."""
    for i in geom_categories[cat]:
        mid = int(mj_model.geom_matid[i])
        if mid >= 0:
            r = orig_mat_rgba[mid]
        else:
            r = orig_geom_rgba[i]
        if r[3] > 0.01:
            return '#{:02x}{:02x}{:02x}'.format(
                int(np.clip(r[0], 0, 1) * 255),
                int(np.clip(r[1], 0, 1) * 255),
                int(np.clip(r[2], 0, 1) * 255),
            )
    return '#888888'

cat_default_hex = {cat: _cat_default_hex(cat) for cat in BODY_CATEGORIES}

print('Geom categories built:')
for cat in BODY_CATEGORIES:
    print(f'  {cat:15s}: {len(geom_categories[cat]):3d} geoms  ({cat_default_hex[cat]})')

# ── Floor geom detection ──────────────────────────────────────────────────────
floor_geom_id = None
floor_mat_id  = None
for _gid in range(mj_model.ngeom):
    if mj_model.geom_type[_gid] == mujoco.mjtGeom.mjGEOM_PLANE:
        floor_geom_id = _gid
        break
if floor_geom_id is None:
    for _gid in range(mj_model.ngeom):
        _gname = mujoco.mj_id2name(mj_model, mujoco.mjtObj.mjOBJ_GEOM, _gid)
        if _gname and any(k in _gname.lower() for k in ('floor', 'ground')):
            floor_geom_id = _gid
            break

if floor_geom_id is not None:
    _mid = int(mj_model.geom_matid[floor_geom_id])
    floor_mat_id    = _mid if _mid >= 0 else None
    orig_floor_rgba = orig_geom_rgba[floor_geom_id].copy()
    print(f'Floor: geom_id={floor_geom_id}  mat_id={floor_mat_id}')
else:
    floor_mat_id    = None
    orig_floor_rgba = np.array([0.5, 0.5, 0.5, 1.0], dtype=np.float32)
    print('No floor plane geom found in model')

if floor_mat_id is not None:
    orig_floor_mat_props = {
        'rgba':        mj_model.mat_rgba[floor_mat_id].copy(),
        'texrepeat':   mj_model.mat_texrepeat[floor_mat_id].copy(),
        'reflectance': float(mj_model.mat_reflectance[floor_mat_id]),
        'shininess':   float(mj_model.mat_shininess[floor_mat_id]),
        'emission':    float(mj_model.mat_emission[floor_mat_id]),
    }
else:
    orig_floor_mat_props = {
        'rgba':        np.ones(4, dtype=np.float32),
        'texrepeat':   np.array([1.0, 1.0]),
        'reflectance': 0.2, 'shininess': 0.5, 'emission': 0.0,
    }

_floor_init_rgb = orig_floor_mat_props['rgba'][:3] if floor_mat_id is not None else orig_floor_rgba[:3]


In [ ]:

# All named cameras defined in the XML
ALL_CAMERAS = [
    'track1', 'track2', 'track3',
    'side', 'side2', 'back', 'bottom', 'bottom2', 'hero',
    'front_left', 'front_right', 'tgt_left', 'tgt_right',
    'eye_left', 'eye_right',
]
HEIGHT, WIDTH = 480, 640

def render_frame(qpos, camera='track1', scene_option=None, show_shadows=True,
                 show_wireframe=False, show_skybox=True, scene_modifiers=None):
    """Render a single frame.

    Args:
        qpos: joint positions array.
        camera: str (named camera) or MjvCamera instance (free-orbit).
        scene_option: MjvOption, or None for bare defaults.
        show_shadows: if False, disables shadow rendering.
        show_wireframe: if True, renders all geometry as wireframe mesh.
        show_skybox: if False, disables the skybox texture (background goes black).
    """
    mj_data.qpos[:] = qpos
    mujoco.mj_forward(mj_model, mj_data)
    if scene_option is None:
        scene_option = mujoco.MjvOption()
    with mujoco.Renderer(mj_model, height=HEIGHT, width=WIDTH) as renderer:
        renderer.update_scene(mj_data, camera=camera, scene_option=scene_option)
        if not show_shadows:
            renderer.scene.flags[mujoco.mjtRndFlag.mjRND_SHADOW] = False
        if show_wireframe:
            renderer.scene.flags[mujoco.mjtRndFlag.mjRND_WIREFRAME] = True
        if not show_skybox:
            renderer.scene.flags[mujoco.mjtRndFlag.mjRND_SKYBOX] = False
        if scene_modifiers:
            for fn, kwargs in scene_modifiers:
                fn(renderer.scene, mj_data, **kwargs)
        return renderer.render()


In [ ]:

# ── Shared visualization state + render helpers ───────────────────────────────
import copy, json

def _dir_to_az_el(d):
    d = np.asarray(d, dtype=float)
    norm = np.linalg.norm(d)
    if norm < 1e-9:
        return 0.0, -45.0
    d = d / norm
    el = float(np.degrees(np.arcsin(np.clip(d[2], -1.0, 1.0))))
    az = float(np.degrees(np.arctan2(d[0], d[1])) % 360)
    return az, el

def _az_el_to_dir(az_deg, el_deg):
    az = np.radians(az_deg); el = np.radians(el_deg)
    return np.array([np.cos(el)*np.sin(az), np.cos(el)*np.cos(az), np.sin(el)])

def _hex_to_rgb(hex_str):
    h = hex_str.lstrip('#')
    return [int(h[i:i+2], 16) / 255.0 for i in (0, 2, 4)]

def _rgb_to_hex(rgb):
    return '#{:02x}{:02x}{:02x}'.format(*[int(np.clip(v, 0, 1) * 255) for v in rgb])

# ── Detach ALL material geoms in categorised set ──────────────────────────────
# Bake mat_rgba * geom_rgba into geom_rgba and clear geom_matid so every geom
# can be independently recoloured via geom_rgba alone.  Rendering-only change.
_categorised_geom_ids = {i for idxs in geom_categories.values() for i in idxs}
for _gi in _categorised_geom_ids:
    _mid = int(mj_model.geom_matid[_gi])
    if _mid < 0:
        continue
    mj_model.geom_rgba[_gi] = np.clip(orig_mat_rgba[_mid] * orig_geom_rgba[_gi], 0.0, 1.0)
    mj_model.geom_matid[_gi] = -1
# Refresh orig_geom_rgba so reset logic sees the baked colours
orig_geom_rgba = mj_model.geom_rgba.copy()

# Recompute cat_default_hex from baked geom_rgba (now the single source of truth)
def _cat_default_hex_baked(cat):
    for i in geom_categories[cat]:
        r = orig_geom_rgba[i]
        if r[3] > 0.01:
            return '#{:02x}{:02x}{:02x}'.format(
                int(np.clip(r[0], 0, 1) * 255),
                int(np.clip(r[1], 0, 1) * 255),
                int(np.clip(r[2], 0, 1) * 255),
            )
    return '#888888'
cat_default_hex = {cat: _cat_default_hex_baked(cat) for cat in BODY_CATEGORIES}

# ── Build per-category geom name lists (visible geoms only, for UI) ───────────
geom_names_by_cat = {}
for cat in BODY_CATEGORIES:
    entries = []
    for gid in geom_categories[cat]:
        alpha = orig_geom_rgba[gid, 3]  # matid is now -1 for all; use geom_rgba
        if alpha < 0.01:
            continue
        gname = mujoco.mj_id2name(mj_model, mujoco.mjtObj.mjOBJ_GEOM, gid) or f'geom_{gid}'
        entries.append((gid, gname))
    geom_names_by_cat[cat] = entries

def _geom_default_hex(gid):
    r = orig_geom_rgba[gid]
    return '#{:02x}{:02x}{:02x}'.format(
        int(np.clip(r[0], 0, 1) * 255),
        int(np.clip(r[1], 0, 1) * 255),
        int(np.clip(r[2], 0, 1) * 255),
    )

# ── Capture skybox gradient init colors from XML defaults ──────────────────────
# rgb1 (.4 .6 .8 → light blue at top) and rgb2 (0 0 0 → black at horizon)
_init_sky_top_hex = _rgb_to_hex([0.4, 0.6, 0.8])  # matches rgb1 in floor.xml
_init_sky_bot_hex = _rgb_to_hex([0.0, 0.0, 0.0])  # matches rgb2 in floor.xml

# Locate the skybox texture so we can overwrite its pixel data at runtime.
_skybox_tex_id = mujoco.mj_name2id(mj_model, mujoco.mjtObj.mjOBJ_TEXTURE, "skybox")

_init_lights = []
for _i in range(min(mj_model.nlight, 3)):
    _az, _el = _dir_to_az_el(mj_model.light_dir[_i])
    _init_lights.append({
        'active':   bool(mj_model.light_active[_i]),
        'ambient':  list(map(float, mj_model.light_ambient[_i])),
        'diffuse':  list(map(float, mj_model.light_diffuse[_i])),
        'specular': list(map(float, mj_model.light_specular[_i])),
        'dir_az': _az, 'dir_el': _el,
    })

vis_state = {
    'colors':      {cat: cat_default_hex[cat] for cat in BODY_CATEGORIES},
    'geom_colors': {},   # {geom_id (int): hex str} — per-geom overrides
    'alpha': 1.0,
    'vis_flags': {
        'contact_points': False, 'contact_forces': False,
        'actuators': False, 'joints': False, 'transparent': False,
        'shadows': True, 'wireframe': False,
    },
    'geom_groups': [True, True, True, True, False, False],
    'site_groups':  [True, True, True, True, True,  False],
    'camera': {
        'mode': 'named', 'named': 'track1',
        'azimuth': 180.0, 'elevation': -30.0, 'distance': 0.3,
        'lookat': [0.0, 0.0, 0.0],
        # MjvCamera type used when mode == 'free':
        #   'free'     → mjCAMERA_FREE     (standard free-orbit)
        #   'fixed'    → mjCAMERA_FIXED    (lock to a named camera; set fixedcamid)
        #   'track'    → mjCAMERA_TRACKING (follow body origin; set trackbody)
        #   'trackcom' → mjCAMERA_TRACKING (follow body CoM;    set trackbody)
        'free_type':  'free',
        'trackbody':  '',       # body name used by track/trackcom modes
        'fixedcamid': '',       # named camera used by 'fixed' type
    },
    'lighting': {
        'lights': _init_lights,
        'use_dual_lighting':   False,
        'use_scale_lights':    False,
        'scale_lights_factor': 1.25,
        'headlight': {
            'active':   bool(mj_model.vis.headlight.active),
            'ambient':  list(map(float, mj_model.vis.headlight.ambient)),
            'diffuse':  list(map(float, mj_model.vis.headlight.diffuse)),
            'specular': list(map(float, mj_model.vis.headlight.specular)),
        },
    },
    'floor': {
        'color':       _rgb_to_hex(_floor_init_rgb),
        'alpha':       float(orig_floor_rgba[3]),
        'texrepeat_x': float(orig_floor_mat_props['texrepeat'][0]),
        'texrepeat_y': float(orig_floor_mat_props['texrepeat'][1]),
        'reflectance': orig_floor_mat_props['reflectance'],
        'shininess':   orig_floor_mat_props['shininess'],
        'emission':    orig_floor_mat_props['emission'],
    },
    'skybox': {
        'show':    True,             # show XML-defined skybox texture
        'sky_top': _init_sky_top_hex,  # gradient top/zenith color (rgb1)
        'sky_bot': _init_sky_bot_hex,  # gradient bottom/horizon color (rgb2)
    },
}

# ── Per-tab reset defaults & initial state snapshots ─────────────────────────
_vis_tab_defaults = {
    'vis_flags': {
        'contact_points': False, 'contact_forces': False,
        'actuators': False, 'joints': False, 'transparent': False,
        'shadows': True, 'wireframe': False,
    },
    'geom_groups': [True, True, True, True, False, False],
    'site_groups':  [True, True, True, True, True,  False],
}
_init_lighting_snapshot = copy.deepcopy(vis_state['lighting'])
_init_camera_snapshot   = copy.deepcopy(vis_state['camera'])
_init_floor_snapshot    = copy.deepcopy(vis_state['floor'])
_init_sky_snapshot      = copy.deepcopy(vis_state['skybox'])

# Set True during batch widget updates to suppress intermediate renders
_suppress_render = False

# Camera presets populated via Settings tab
saved_camera_presets = {}

# ── Helper functions ──────────────────────────────────────────────────────────

def apply_geom_colors():
    alpha = vis_state['alpha']
    geom_overrides = vis_state['geom_colors']  # {geom_id: hex}
    for cat, idxs in geom_categories.items():
        cat_rgb = _hex_to_rgb(vis_state['colors'][cat])
        for i in idxs:
            if orig_geom_rgba[i, 3] < 0.01:
                continue   # invisible geom (collision shape etc.) — leave alone
            rgb = _hex_to_rgb(geom_overrides[i]) if i in geom_overrides else cat_rgb
            mj_model.geom_rgba[i, :3] = rgb
            mj_model.geom_rgba[i,  3] = orig_geom_rgba[i, 3] * alpha

def apply_lighting():
    for i, ld in enumerate(vis_state['lighting']['lights']):
        if i >= mj_model.nlight:
            break
        mj_model.light_active[i]   = int(ld['active'])
        mj_model.light_ambient[i]  = ld['ambient']
        mj_model.light_diffuse[i]  = ld['diffuse']
        mj_model.light_specular[i] = ld['specular']
        mj_model.light_dir[i]      = _az_el_to_dir(ld['dir_az'], ld['dir_el'])
    hl = vis_state['lighting']['headlight']
    mj_model.vis.headlight.active      = int(hl['active'])
    mj_model.vis.headlight.ambient[:]  = hl['ambient']
    mj_model.vis.headlight.diffuse[:]  = hl['diffuse']
    mj_model.vis.headlight.specular[:] = hl['specular']

def apply_floor_props():
    if floor_geom_id is None:
        return
    fld = vis_state['floor']
    rgb = _hex_to_rgb(fld['color'])
    mj_model.geom_rgba[floor_geom_id] = [*rgb, fld['alpha']]
    if floor_mat_id is not None:
        mj_model.mat_rgba[floor_mat_id]        = [*rgb, fld['alpha']]
        mj_model.mat_texrepeat[floor_mat_id]   = [fld['texrepeat_x'], fld['texrepeat_y']]
        mj_model.mat_reflectance[floor_mat_id] = fld['reflectance']
        mj_model.mat_shininess[floor_mat_id]   = fld['shininess']
        mj_model.mat_emission[floor_mat_id]    = fld['emission']

def _make_sky_pixels(top_rgb, bot_rgb):
    """Build cube-map gradient pixel data for the skybox texture.
    In MuJoCo tex_data, a skybox cube-map is stored as a (6*face_h) × face_w
    strip: tex_height = 6 * face_h, tex_width = face_w.
    Uses elevation-based gradient: top=zenith (rgb1), bot=horizon (rgb2).
    """
    if _skybox_tex_id < 0:
        return None
    total_h = int(mj_model.tex_height[_skybox_tex_id])
    w       = int(mj_model.tex_width[_skybox_tex_id])
    face_h  = max(1, total_h // 6)  # each face is face_h × w
    top = np.array(top_rgb, dtype=np.float64)
    bot = np.array(bot_rgb, dtype=np.float64)
    # Cube-face geometry: (face_normal, right_vec, up_vec) MuJoCo face order
    face_axes = [
        (np.array([1.,0.,0.]),  np.array([0.,0.,-1.]), np.array([0.,-1.,0.])),
        (np.array([-1.,0.,0.]), np.array([0.,0.,1.]),  np.array([0.,-1.,0.])),
        (np.array([0.,1.,0.]),  np.array([1.,0.,0.]),  np.array([0.,0.,1.])),
        (np.array([0.,-1.,0.]), np.array([1.,0.,0.]),  np.array([0.,0.,-1.])),
        (np.array([0.,0.,1.]),  np.array([1.,0.,0.]),  np.array([0.,-1.,0.])),
        (np.array([0.,0.,-1.]), np.array([-1.,0.,0.]), np.array([0.,-1.,0.])),
    ]
    pixels = np.zeros((total_h * w, 3), dtype=np.uint8)  # full cube-map strip
    rows = np.arange(face_h); cols = np.arange(w)
    v_arr = 1.0 - 2.0 * (rows + 0.5) / face_h  # +1 at top, -1 at bottom
    u_arr = -1.0 + 2.0 * (cols + 0.5) / w
    V, U = np.meshgrid(v_arr, u_arr, indexing="ij")  # (face_h, w)
    for fi, (norm, right, up) in enumerate(face_axes):
        d = norm + U[:,:,None] * right + V[:,:,None] * up
        d /= np.linalg.norm(d, axis=2, keepdims=True)
        t = np.clip(0.5 + 0.5 * d[:,:,1], 0.0, 1.0)
        color = (1.0 - t[:,:,None]) * bot + t[:,:,None] * top
        pixels[fi*face_h*w:(fi+1)*face_h*w] = np.clip(color * 255, 0, 255).astype(np.uint8).reshape(-1, 3)
    return pixels

def apply_sky_props():
    if _skybox_tex_id < 0:
        return
    sky = vis_state["skybox"]
    pixels = _make_sky_pixels(_hex_to_rgb(sky["sky_top"]), _hex_to_rgb(sky["sky_bot"]))
    if pixels is None:
        return
    h = int(mj_model.tex_height[_skybox_tex_id])
    w = int(mj_model.tex_width[_skybox_tex_id])
    adr = int(mj_model.tex_adr[_skybox_tex_id])
    nchan = int(mj_model.tex_nchannel[_skybox_tex_id]) if hasattr(mj_model, "tex_nchannel") else 3
    if nchan == 4:
        rgba = np.ones((len(pixels), 4), dtype=np.uint8) * 255
        rgba[:, :3] = pixels
        flat = rgba.flatten()
    else:
        flat = pixels.flatten()
    # MuJoCo 3.x renamed tex_rgb → tex_data; support both
    _tex_buf = getattr(mj_model, "tex_data", None)
    if _tex_buf is None: _tex_buf = getattr(mj_model, "tex_rgb", None)
    if _tex_buf is not None:
        _tex_buf[adr:adr + len(flat)] = flat


def make_scene_option():
    opt = mujoco.MjvOption()
    f = vis_state['vis_flags']
    opt.flags[mujoco.mjtVisFlag.mjVIS_CONTACTPOINT] = f['contact_points']
    opt.flags[mujoco.mjtVisFlag.mjVIS_CONTACTFORCE] = f['contact_forces']
    opt.flags[mujoco.mjtVisFlag.mjVIS_ACTUATOR]     = f['actuators']
    opt.flags[mujoco.mjtVisFlag.mjVIS_JOINT]        = f['joints']
    opt.flags[mujoco.mjtVisFlag.mjVIS_TRANSPARENT]  = f['transparent']
    for k in range(min(6, len(opt.geomgroup))):
        opt.geomgroup[k] = int(vis_state['geom_groups'][k])
    for k in range(min(6, len(opt.sitegroup))):
        opt.sitegroup[k] = int(vis_state['site_groups'][k])
    return opt

# ── Camera type map ───────────────────────────────────────────────────────────
# Maps free_type string → (mjtCamera constant, needs_trackbody, needs_fixedcam)
# MuJoCo 3.4 provides: FREE, TRACKING, FIXED, USER.
_FREE_TYPE_MAP = {
    'free':     (mujoco.mjtCamera.mjCAMERA_FREE,     False, False),
    'fixed':    (mujoco.mjtCamera.mjCAMERA_FIXED,    False, True),
    'track':    (mujoco.mjtCamera.mjCAMERA_TRACKING, True,  False),
    'trackcom': (mujoco.mjtCamera.mjCAMERA_TRACKING, True,  False),
}

def make_camera():
    """Build a MuJoCo camera from vis_state['camera'].

    Named mode  → returns the camera name string; MuJoCo uses its XML-defined
                  mode (fixed/track/etc. as set in the XML).
    Free mode   → builds an MjvCamera with the selected free_type:
        free      – standard free-orbit (azimuth/elevation/distance/lookat)
        fixed     – locks to a named XML camera position (set fixedcamid)
        track     – camera origin follows a body (set trackbody)
        trackcom  – camera origin follows a body's CoM (set trackbody)
    """
    c = vis_state['camera']
    if c['mode'] == 'named':
        return c['named']

    free_type = c.get('free_type', 'free')
    mj_type, needs_body, needs_fixedcam = _FREE_TYPE_MAP.get(free_type, _FREE_TYPE_MAP['free'])

    cam = mujoco.MjvCamera()
    cam.type      = mj_type
    cam.azimuth   = c['azimuth']
    cam.elevation = c['elevation']
    cam.distance  = c['distance']
    cam.lookat[:] = c['lookat']

    if needs_fixedcam:
        cam_id = mujoco.mj_name2id(mj_model, mujoco.mjtObj.mjOBJ_CAMERA, c.get('fixedcamid', ''))
        cam.fixedcamid = max(cam_id, 0)

    if needs_body:
        body_id = mujoco.mj_name2id(mj_model, mujoco.mjtObj.mjOBJ_BODY, c.get('trackbody', ''))
        cam.trackbodyid = max(body_id, 0)

    return cam
out = widgets.Output()
current_qpos = mj_model.qpos_spring.copy()
current_qpos[wing_joint_idxs] = 0.0
current_qpos[abd_idxs] = 0.0

# ── Lighting preset functions (scene-level, run after update_scene) ─────────────────────
def dual_lighting(scene, state, **kwargs):
    geom_xpos = kwargs.get('geom_xpos')
    if geom_xpos is None and state is not None:
        geom_xpos = state.geom_xpos
    if geom_xpos is None:
        return
    body_pos = geom_xpos[1]
    if scene.nlight > 2:
        scene.lights[0].pos[:] = body_pos + np.array([0.6, 0.6, 0.0])
        scene.lights[0].dir[:] = body_pos - scene.lights[2].pos
    scene.lights[0].diffuse[:] = [1.4, 1.3, 1.0]
    scene.lights[0].specular[:] = [1.4, 1.4, 1.4]
    scene.lights[0].cutoff = 20.0
    scene.lights[0].exponent = 3.0
    scene.lights[0].attenuation[:] = [1, 0.0, 0.0]

def scale_lights(scene, state, scale=1.25, **kwargs):
    for i in range(scene.nlight):
        scene.lights[i].diffuse[:] *= scale
        scene.lights[i].ambient[:] *= scale
        scene.lights[i].specular[:] = 0

def _build_scene_modifiers():
    mods = []
    if vis_state['lighting'].get('use_dual_lighting', False):
        mods.append((dual_lighting, {}))
    if vis_state['lighting'].get('use_scale_lights', False):
        mods.append((scale_lights, {'scale': vis_state['lighting'].get('scale_lights_factor', 1.25)}))
    return mods

def do_render():
    if _suppress_render:
        return
    apply_geom_colors()
    apply_lighting()
    apply_floor_props()
    apply_sky_props()
    frame = render_frame(
        current_qpos, camera=make_camera(), scene_option=make_scene_option(),
        show_shadows=vis_state['vis_flags']['shadows'],
        show_wireframe=vis_state['vis_flags']['wireframe'],
        show_skybox=vis_state['skybox']['show'],
        scene_modifiers=_build_scene_modifiers(),
    )
    # Clear the output widget immediately (not deferred) so that multiple rapid
    # renders — e.g. when loading a settings file with saved camera presets —
    # never accumulate.  Using out.clear_output() directly avoids the
    # `clear_output(wait=True)` deferred-clear behaviour that can leave several
    # images visible when renders arrive faster than the frontend can process them.
    out.clear_output(wait=False)
    with out:
        media.show_image(frame)

# ── Save settings to JSON ─────────────────────────────────────────────────────
def save_settings(filepath):
    # geom_colors keys are ints; JSON requires string keys
    geom_colors_str = {str(k): v for k, v in vis_state['geom_colors'].items()}
    # Snapshot per-geom rgba after applying current colors — lets apply_render_settings
    # reproduce the exact render from JSON without needing the interactive notebook.
    apply_geom_colors()
    _all_cat_ids = sorted({i for idxs in geom_categories.values() for i in idxs})
    geom_render_state = {str(i): list(map(float, mj_model.geom_rgba[i])) for i in _all_cat_ids}
    data = {
        'colors':           vis_state['colors'],
        'geom_colors':      geom_colors_str,
        'alpha':            vis_state['alpha'],
        'vis_flags':        dict(vis_state['vis_flags']),
        'geom_groups':      vis_state['geom_groups'][:],
        'site_groups':      vis_state['site_groups'][:],
        'camera':           copy.deepcopy(vis_state['camera']),
        'lighting':         copy.deepcopy(vis_state['lighting']),
        'floor':            copy.deepcopy(vis_state['floor']),
        'skybox':           copy.deepcopy(vis_state['skybox']),
        'geom_render_state': geom_render_state,  # for apply_render_settings()
        'camera_presets':   saved_camera_presets,
    }
    with open(filepath, 'w') as f:
        json.dump(data, f, indent=2)

# ── Standalone render helper ──────────────────────────────────────────────────
def apply_render_settings(model, settings):
    """Apply saved pose_tuner settings to a MuJoCo model for scripted rendering.

    Lets you reproduce a saved view without the interactive notebook:
        import json, mujoco
        model = mujoco.MjModel.from_xml_path('...')
        data  = mujoco.MjData(model)
        with open('pose_tuner_settings.json') as f:
            s = json.load(f)
        cam = apply_render_settings(model, s)
        with mujoco.Renderer(model, 480, 640) as r:
            r.update_scene(data, camera=cam)
            frame = r.render()

    Args:
        model:    mujoco.MjModel instance (modified in-place).
        settings: dict loaded from a pose_tuner JSON, or a file path str.

    Returns:
        camera: named camera str, or MjvCamera instance (for free-orbit saves).
    """
    if isinstance(settings, str):
        with open(settings) as _f:
            settings = json.load(_f)

    # ── Per-geom colors ───────────────────────────────────────────────────────
    # geom_render_state contains baked + colored rgba for every categorised geom.
    for gid_str, rgba in settings.get('geom_render_state', {}).items():
        gid = int(gid_str)
        model.geom_matid[gid] = -1          # detach material (same as baking step)
        model.geom_rgba[gid]  = rgba

    # ── Floor ─────────────────────────────────────────────────────────────────
    fld = settings.get('floor', {})
    if fld:
        floor_gid = next(
            (i for i in range(model.ngeom) if model.geom_type[i] == mujoco.mjtGeom.mjGEOM_PLANE),
            None,
        )
        if floor_gid is not None:
            rgb = _hex_to_rgb(fld.get('color', '#888888'))
            a   = fld.get('alpha', 1.0)
            model.geom_rgba[floor_gid] = [*rgb, a]
            mid = int(model.geom_matid[floor_gid])
            if mid >= 0:
                model.mat_rgba[mid]        = [*rgb, a]
                model.mat_texrepeat[mid]   = [fld.get('texrepeat_x', 1.0), fld.get('texrepeat_y', 1.0)]
                model.mat_reflectance[mid] = fld.get('reflectance', 0.2)
                model.mat_shininess[mid]   = fld.get('shininess', 0.5)
                model.mat_emission[mid]    = fld.get('emission', 0.0)

    # ── Lighting ──────────────────────────────────────────────────────────────
    for i, ld in enumerate(settings.get('lighting', {}).get('lights', [])):
        if i >= model.nlight:
            break
        model.light_active[i]   = int(ld.get('active', True))
        model.light_ambient[i]  = ld.get('ambient',  [0.0, 0.0, 0.0])
        model.light_diffuse[i]  = ld.get('diffuse',  [0.8, 0.8, 0.8])
        model.light_specular[i] = ld.get('specular', [0.3, 0.3, 0.3])
        model.light_dir[i]      = _az_el_to_dir(ld.get('dir_az', 0.0), ld.get('dir_el', -45.0))
    hl = settings.get('lighting', {}).get('headlight', {})
    if hl:
        model.vis.headlight.active      = int(hl.get('active', True))
        model.vis.headlight.ambient[:]  = hl.get('ambient',  [0.0, 0.0, 0.0])
        model.vis.headlight.diffuse[:]  = hl.get('diffuse',  [0.4, 0.4, 0.4])
        model.vis.headlight.specular[:] = hl.get('specular', [0.0, 0.0, 0.0])

    # ── Skybox texture gradient ──────────────────────────────────────────────
    sky = settings.get('skybox', {})
    if sky and _skybox_tex_id >= 0:
        _old = vis_state["skybox"]
        vis_state["skybox"] = sky
        apply_sky_props()
        vis_state["skybox"] = _old

    # ── Camera ────────────────────────────────────────────────────────────────
    cam_cfg = settings.get('camera', {'mode': 'named', 'named': 'track1'})
    if cam_cfg.get('mode', 'named') == 'named':
        return cam_cfg.get('named', 'track1')

    # Free-camera mode — reconstruct MjvCamera respecting saved free_type.
    free_type = cam_cfg.get('free_type', 'free')
    mj_type, needs_body, needs_fixedcam = _FREE_TYPE_MAP.get(free_type, _FREE_TYPE_MAP['free'])

    cam = mujoco.MjvCamera()
    cam.type      = mj_type
    cam.azimuth   = cam_cfg.get('azimuth',   180.0)
    cam.elevation = cam_cfg.get('elevation', -30.0)
    cam.distance  = cam_cfg.get('distance',    0.3)
    cam.lookat[:] = cam_cfg.get('lookat',   [0.0, 0.0, 0.0])

    if needs_fixedcam:
        cam_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_CAMERA, cam_cfg.get('fixedcamid', ''))
        cam.fixedcamid = max(cam_id, 0)

    if needs_body:
        body_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, cam_cfg.get('trackbody', ''))
        cam.trackbodyid = max(body_id, 0)

    return cam


# GUI Interface

In [ ]:

# ── 7-Tab interactive UI ──────────────────────────────────────────────────────
base_qpos = mj_model.qpos_spring.copy()

# ════════════════════════════════════════════════════════════════════════════════
# Tab 0 — Pose
# ════════════════════════════════════════════════════════════════════════════════
def _get_leg_key(name):
    for leg in ['T1', 'T2', 'T3']:
        if leg in name:
            side = 'left' if 'left' in name else 'right'
            return f'{leg}_{side}'
    return 'other'

leg_groups = {}
for name in leg_joints:
    key = _get_leg_key(name)
    leg_groups.setdefault(key, []).append(name)

pose_sliders = {}
for name, idx in leg_joints.items():
    default = float(base_qpos[idx])
    pose_sliders[name] = widgets.FloatSlider(
        value=default, min=round(default - 1.8, 2), max=round(default + 1.8, 2),
        step=0.01,
        description=(name.replace('_left','').replace('_right','')
                        .replace('_T1','').replace('_T2','').replace('_T3',''))[:20],
        continuous_update=False, readout_format='.2f',
        layout=widgets.Layout(width='550px'), style={'description_width': '160px'},
    )

reset_pose_btn = widgets.Button(description='Reset to springref', button_style='warning')

def _on_pose_slider(change):
    current_qpos[leg_joints[change['owner']._joint_name]] = change['new']
    do_render()

def _on_reset_pose(b):
    global _suppress_render
    _suppress_render = True
    current_qpos[:] = base_qpos
    for name, sl in pose_sliders.items():
        sl.unobserve_all()
        sl.value = float(base_qpos[leg_joints[name]])
        sl._joint_name = name
        sl.observe(_on_pose_slider, names='value')
    _suppress_render = False
    do_render()

for name, sl in pose_sliders.items():
    sl._joint_name = name
    sl.observe(_on_pose_slider, names='value')
reset_pose_btn.on_click(_on_reset_pose)

_leg_tab_order    = [k for k in ['T1_left','T1_right','T2_left','T2_right','T3_left','T3_right'] if k in leg_groups]
_leg_tab_children = [widgets.VBox([pose_sliders[n] for n in leg_groups[k]]) for k in _leg_tab_order]
_leg_tabs = widgets.Tab(children=_leg_tab_children)
for i, k in enumerate(_leg_tab_order):
    _leg_tabs.set_title(i, k)

tab_pose = widgets.VBox([reset_pose_btn, _leg_tabs])

# ════════════════════════════════════════════════════════════════════════════════
# Tab 1 — Colors
# Layout per category:
#   [Category color picker]  [Geom dropdown]  [Geom color picker (hidden if All)]
# Category picker sets whole body; dropdown selects individual geom for override.
# ════════════════════════════════════════════════════════════════════════════════

_ALL_LABEL = '— All —'

# Dicts populated in loop below
color_pickers     = {}   # cat -> category ColorPicker
geom_dd           = {}   # cat -> Dropdown of geom names
geom_picker       = {}   # cat -> per-geom ColorPicker
geom_picker_row   = {}   # cat -> HBox(geom_dd, geom_picker)
_cat_geom_map     = {}   # cat -> [(geom_id, name_str), ...]
_geom_picker_cb   = {}   # cat -> the _on_geom_picker callback (for re-attaching)
_geom_hex_cb      = {}   # cat -> the _on_geom_hex callback (for re-attaching)
geom_hex_txt      = {}   # cat -> per-geom Text widget for typed hex input

for cat in BODY_CATEGORIES:
    entries = geom_names_by_cat.get(cat, [])
    _cat_geom_map[cat] = entries

    # ── 1. Category-level picker ───────────────────────────────────────────────
    cp = widgets.ColorPicker(
        value=cat_default_hex[cat], description=cat, concise=False,
        layout=widgets.Layout(width='300px'), style={'description_width': '110px'},
    )
    def _on_cat_color(change, cat=cat):
        vis_state['colors'][cat] = change['new']
        do_render()
    cp.observe(_on_cat_color, names='value')
    color_pickers[cat] = cp

    # ── 2. Geom dropdown ──────────────────────────────────────────────────────
    dd_options = [_ALL_LABEL] + [lbl for _, lbl in entries]
    dd = widgets.Dropdown(
        options=dd_options, value=_ALL_LABEL,
        layout=widgets.Layout(width='210px'),
    )

    # ── 3. Per-geom picker + hex text input (initially hidden) ────────────────
    gp = widgets.ColorPicker(
        value=cat_default_hex[cat], description='', concise=True,
        layout=widgets.Layout(width='60px', display='none'),
    )
    gt = widgets.Text(
        value=cat_default_hex[cat], placeholder='#rrggbb',
        layout=widgets.Layout(width='86px', display='none'),
    )

    # ── Callbacks (capturing cat / dd / gp / gt in default-arg closure) ───────
    def _on_geom_picker(change, cat=cat, dd=dd, gt=gt):
        sel = dd.value
        if sel == _ALL_LABEL:
            return
        gid = next((g for g, lbl in _cat_geom_map[cat] if lbl == sel), None)
        if gid is not None:
            vis_state['geom_colors'][gid] = change['new']
            gt.unobserve_all()
            gt.value = change['new']
            gt.observe(_geom_hex_cb[cat], names='value')
            do_render()

    def _on_geom_hex(change, cat=cat, dd=dd, gp=gp):
        """Apply a manually typed hex colour (#RRGGBB) to the selected geom."""
        raw = change['new'].strip()
        if not raw.startswith('#'):
            raw = '#' + raw
        if len(raw) != 7:
            return
        try:
            int(raw[1:], 16)
        except ValueError:
            return
        sel = dd.value
        if sel == _ALL_LABEL:
            return
        gid = next((g for g, lbl in _cat_geom_map[cat] if lbl == sel), None)
        if gid is not None:
            vis_state['geom_colors'][gid] = raw
            gp.unobserve_all()
            gp.value = raw
            gp.observe(_geom_picker_cb[cat], names='value')
            do_render()

    def _on_geom_dd(change, cat=cat, dd=dd, gp=gp, gt=gt):
        sel = change['new']
        if sel == _ALL_LABEL:
            gp.layout.display = 'none'
            gt.layout.display = 'none'
        else:
            gid = next((g for g, lbl in _cat_geom_map[cat] if lbl == sel), None)
            if gid is not None:
                hex_val = vis_state['geom_colors'].get(gid, vis_state['colors'][cat])
                gp.unobserve_all()
                gp.value = hex_val
                gp.observe(_geom_picker_cb[cat], names='value')
                gt.unobserve_all()
                gt.value = hex_val
                gt.observe(_geom_hex_cb[cat], names='value')
            gp.layout.display = ''
            gt.layout.display = ''

    _geom_picker_cb[cat] = _on_geom_picker
    _geom_hex_cb[cat]    = _on_geom_hex

    dd.observe(_on_geom_dd,     names='value')
    gp.observe(_on_geom_picker, names='value')
    gt.observe(_on_geom_hex,    names='value')

    geom_dd[cat]         = dd
    geom_picker[cat]     = gp
    geom_hex_txt[cat]    = gt
    geom_picker_row[cat] = widgets.HBox(
        [dd, gp, gt], layout=widgets.Layout(align_items='center'),
    )

alpha_slider = widgets.FloatSlider(
    value=1.0, min=0.0, max=1.0, step=0.05, description='Global alpha',
    continuous_update=False, readout_format='.2f',
    layout=widgets.Layout(width='420px'), style={'description_width': '110px'},
)
def _on_alpha(change):
    vis_state['alpha'] = change['new']; do_render()
alpha_slider.observe(_on_alpha, names='value')

reset_colors_btn = widgets.Button(description='Reset colors', button_style='warning')
def _on_reset_colors(b):
    global _suppress_render
    _suppress_render = True
    vis_state['geom_colors'].clear()
    for cat in BODY_CATEGORIES:
        vis_state['colors'][cat] = cat_default_hex[cat]
        color_pickers[cat].value = cat_default_hex[cat]
        # Reset geom dropdown to All → hides geom picker
        geom_dd[cat].value = _ALL_LABEL
        # Re-arm geom picker with default color (it's hidden anyway)
        geom_hex_txt[cat].unobserve_all()
        geom_hex_txt[cat].value = cat_default_hex[cat]
        geom_hex_txt[cat].layout.display = 'none'
        geom_hex_txt[cat].observe(_geom_hex_cb[cat], names='value')
    alpha_slider.value = 1.0
    _suppress_render = False
    mj_model.geom_rgba[:] = orig_geom_rgba
    mj_model.mat_rgba[:]  = orig_mat_rgba
    do_render()
reset_colors_btn.on_click(_on_reset_colors)

# Assemble color rows: one per category
_color_rows = []
for cat in BODY_CATEGORIES:
    _color_rows.append(widgets.HBox(
        [color_pickers[cat], geom_picker_row[cat]],
        layout=widgets.Layout(align_items='center', margin='1px 0'),
    ))

tab_colors = widgets.VBox([
    widgets.HBox([reset_colors_btn, alpha_slider]),
    widgets.HTML('<small style="color:#666">'
                 'Left picker: whole body color. Dropdown: select a geom for individual override.</small>'),
    *_color_rows,
])

# ════════════════════════════════════════════════════════════════════════════════
# Tab 2 — Visualization
# ════════════════════════════════════════════════════════════════════════════════
_VIS_FLAGS = [
    ('contact_points', 'Contact Points'), ('contact_forces', 'Contact Forces'),
    ('actuators', 'Actuators'), ('joints', 'Joints'),
    ('transparent', 'Transparent'), ('shadows', 'Shadows'), ('wireframe', 'Wireframe'),
]
vis_checkboxes = {}
for key, label in _VIS_FLAGS:
    cb = widgets.Checkbox(value=vis_state['vis_flags'][key], description=label,
                          layout=widgets.Layout(width='190px'))
    def _on_vis_flag(change, key=key):
        vis_state['vis_flags'][key] = change['new']; do_render()
    cb.observe(_on_vis_flag, names='value')
    vis_checkboxes[key] = cb

geom_group_btns = []
for g in range(6):
    btn = widgets.ToggleButton(value=vis_state['geom_groups'][g], description=f'Geom {g}',
                               button_style='info' if vis_state['geom_groups'][g] else '',
                               layout=widgets.Layout(width='90px'))
    def _on_geom_grp(change, g=g):
        vis_state['geom_groups'][g] = change['new']
        change['owner'].button_style = 'info' if change['new'] else ''; do_render()
    btn.observe(_on_geom_grp, names='value')
    geom_group_btns.append(btn)

site_group_btns = []
for g in range(6):
    btn = widgets.ToggleButton(value=vis_state['site_groups'][g], description=f'Site {g}',
                               button_style='info' if vis_state['site_groups'][g] else '',
                               layout=widgets.Layout(width='90px'))
    def _on_site_grp(change, g=g):
        vis_state['site_groups'][g] = change['new']
        change['owner'].button_style = 'info' if change['new'] else ''; do_render()
    btn.observe(_on_site_grp, names='value')
    site_group_btns.append(btn)

reset_vis_btn = widgets.Button(description='Reset visualization', button_style='warning')
def _on_reset_vis(b):
    global _suppress_render
    _suppress_render = True
    for key, dflt in _vis_tab_defaults['vis_flags'].items():
        vis_state['vis_flags'][key] = dflt
        vis_checkboxes[key].value = dflt
    for g, dflt in enumerate(_vis_tab_defaults['geom_groups']):
        vis_state['geom_groups'][g] = dflt
        geom_group_btns[g].value = dflt
        geom_group_btns[g].button_style = 'info' if dflt else ''
    for g, dflt in enumerate(_vis_tab_defaults['site_groups']):
        vis_state['site_groups'][g] = dflt
        site_group_btns[g].value = dflt
        site_group_btns[g].button_style = 'info' if dflt else ''
    _suppress_render = False
    do_render()
reset_vis_btn.on_click(_on_reset_vis)

tab_vis = widgets.VBox([
    widgets.HBox([reset_vis_btn]),
    widgets.HTML('<b>Visualization Flags</b>'),
    widgets.GridBox([vis_checkboxes[k] for k, _ in _VIS_FLAGS],
                    layout=widgets.Layout(grid_template_columns='repeat(3, 200px)')),
    widgets.HTML('<br><b>Geom Groups</b>'),
    widgets.HBox(geom_group_btns),
    widgets.HTML('<b>Site Groups</b>'),
    widgets.HBox(site_group_btns),
])

# ════════════════════════════════════════════════════════════════════════════════
# Tab 3 — Lighting
# ════════════════════════════════════════════════════════════════════════════════
_CHANNELS = ['R', 'G', 'B']

def _make_rgb_sliders(label, init_rgb, on_change_fn):
    sls = []
    for ch in range(3):
        sl = widgets.FloatSlider(
            value=float(init_rgb[ch]), min=0.0, max=1.0, step=0.01,
            description=f'{label} {_CHANNELS[ch]}', continuous_update=False,
            readout_format='.2f', layout=widgets.Layout(width='420px'),
            style={'description_width': '100px'},
        )
        def _cb(change, ch=ch): on_change_fn(ch, change['new'])
        sl.observe(_cb, names='value')
        sls.append(sl)
    return sls
_LIGHT_NAMES = ['Right', 'Left', 'Tracking']
_light_panels = []
lighting_widgets = {}  # {li or 'headlight': widget refs for reset/load}

for li, lname in enumerate(_LIGHT_NAMES):
    ld = vis_state['lighting']['lights'][li]
    active_cb = widgets.Checkbox(value=ld['active'], description='Active',
                                 layout=widgets.Layout(width='150px'))
    def _on_light_active(change, li=li):
        vis_state['lighting']['lights'][li]['active'] = change['new']; do_render()
    active_cb.observe(_on_light_active, names='value')
    def _amb(ch, val, li=li):  vis_state['lighting']['lights'][li]['ambient'][ch]  = val; do_render()
    def _diff(ch, val, li=li): vis_state['lighting']['lights'][li]['diffuse'][ch]  = val; do_render()
    def _spec(ch, val, li=li): vis_state['lighting']['lights'][li]['specular'][ch] = val; do_render()
    amb_sls  = _make_rgb_sliders('Ambient',  ld['ambient'],  _amb)
    diff_sls = _make_rgb_sliders('Diffuse',  ld['diffuse'],  _diff)
    spec_sls = _make_rgb_sliders('Specular', ld['specular'], _spec)

    az_sl = widgets.FloatSlider(value=ld['dir_az'], min=0.0, max=360.0, step=1.0,
                                description='Dir Azimuth', continuous_update=False,
                                style={'description_width': '100px'})
    el_sl = widgets.FloatSlider(value=ld['dir_el'], min=-90.0, max=0.0, step=1.0,
                                description='Dir Elevation', continuous_update=False,
                                readout_format='.0f', layout=widgets.Layout(width='420px'),
                                style={'description_width': '100px'})
    def _on_az(change, li=li): vis_state['lighting']['lights'][li]['dir_az'] = change['new']; do_render()
    def _on_el(change, li=li): vis_state['lighting']['lights'][li]['dir_el'] = change['new']; do_render()
    az_sl.observe(_on_az, names='value')
    el_sl.observe(_on_el, names='value')

    lighting_widgets[li] = {
        'active': active_cb, 'ambient': amb_sls, 'diffuse': diff_sls,
        'specular': spec_sls, 'dir_az': az_sl, 'dir_el': el_sl,
    }
    _light_panels.append(widgets.VBox([active_cb] + amb_sls + diff_sls + spec_sls + [az_sl, el_sl]))

# Headlight
_hl = vis_state['lighting']['headlight']
hl_active_cb = widgets.Checkbox(value=_hl['active'], description='Active',
                                layout=widgets.Layout(width='150px'))
def _on_hl_active(change): vis_state['lighting']['headlight']['active'] = change['new']; do_render()
hl_active_cb.observe(_on_hl_active, names='value')
def _hl_amb(ch, val):  vis_state['lighting']['headlight']['ambient'][ch]  = val; do_render()
def _hl_diff(ch, val): vis_state['lighting']['headlight']['diffuse'][ch]  = val; do_render()
def _hl_spec(ch, val): vis_state['lighting']['headlight']['specular'][ch] = val; do_render()

hl_amb_sls  = _make_rgb_sliders('Ambient',  _hl['ambient'],  _hl_amb)
hl_diff_sls = _make_rgb_sliders('Diffuse',  _hl['diffuse'],  _hl_diff)
hl_spec_sls = _make_rgb_sliders('Specular', _hl['specular'], _hl_spec)
lighting_widgets['headlight'] = {
    'active': hl_active_cb, 'ambient': hl_amb_sls,
    'diffuse': hl_diff_sls, 'specular': hl_spec_sls,
}
_light_panels.append(widgets.VBox([hl_active_cb] + hl_amb_sls + hl_diff_sls + hl_spec_sls))


# ── Lighting preset toggles ─────────────────────────────────────────────────
dual_light_cb = widgets.Checkbox(value=False, description='Dual lighting',
                                  layout=widgets.Layout(width='188px'))
scale_light_cb = widgets.Checkbox(value=False, description='Scale lights',
                                   layout=widgets.Layout(width='168px'))
scale_sl = widgets.FloatSlider(
    value=1.25, min=0.1, max=3.0, step=0.05, description='Scale',
    continuous_update=False, readout_format='.2f',
    layout=widgets.Layout(width='300px', display='none'),
    style={'description_width': '60px'},
)

def _on_dual_light(change):
    vis_state['lighting']['use_dual_lighting'] = change['new']; do_render()
def _on_scale_light(change):
    vis_state['lighting']['use_scale_lights'] = change['new']
    scale_sl.layout.display = '' if change['new'] else 'none'
    do_render()
def _on_scale_sl(change):
    vis_state['lighting']['scale_lights_factor'] = change['new']; do_render()
dual_light_cb.observe(_on_dual_light, names='value')
scale_light_cb.observe(_on_scale_light, names='value')
scale_sl.observe(_on_scale_sl, names='value')

reset_lighting_btn = widgets.Button(description='Reset lighting', button_style='warning')
def _on_reset_lighting(b):
    global _suppress_render
    _suppress_render = True
    for li, ld in enumerate(_init_lighting_snapshot['lights']):
        vis_state['lighting']['lights'][li] = copy.deepcopy(ld)
        lw = lighting_widgets[li]
        lw['active'].value = ld['active']
        for ch in range(3):
            lw['ambient'][ch].value  = ld['ambient'][ch]
            lw['diffuse'][ch].value  = ld['diffuse'][ch]
            lw['specular'][ch].value = ld['specular'][ch]
        lw['dir_az'].value = ld['dir_az']
        lw['dir_el'].value = ld['dir_el']
    hl = _init_lighting_snapshot['headlight']
    vis_state['lighting']['headlight'] = copy.deepcopy(hl)
    hw = lighting_widgets['headlight']
    hw['active'].value = hl['active']
    for ch in range(3):
        hw['ambient'][ch].value  = hl['ambient'][ch]
        hw['diffuse'][ch].value  = hl['diffuse'][ch]
        hw['specular'][ch].value = hl['specular'][ch]
    vis_state['lighting']['use_dual_lighting'] = False
    vis_state['lighting']['use_scale_lights'] = False
    vis_state['lighting']['scale_lights_factor'] = 1.25
    dual_light_cb.value = False
    scale_light_cb.value = False
    scale_sl.value = 1.25
    scale_sl.layout.display = 'none'
    _suppress_render = False
    do_render()
reset_lighting_btn.on_click(_on_reset_lighting)

_lighting_accordion = widgets.Accordion(children=_light_panels)
for i, name in enumerate(_LIGHT_NAMES + ['Headlight']):
    _lighting_accordion.set_title(i, name)
_lighting_accordion.selected_index = None
_preset_row = widgets.HBox(
    [dual_light_cb, scale_light_cb, scale_sl],
    layout=widgets.Layout(align_items='center', margin='4px 0'),
)
tab_lighting = widgets.VBox([widgets.HBox([reset_lighting_btn]), _preset_row, _lighting_accordion])

# ════════════════════════════════════════════════════════════════════════════════
# Tab 4 — Camera
# ════════════════════════════════════════════════════════════════════════════════
cam_mode_toggle = widgets.ToggleButtons(options=['Named', 'Free orbit'], value='Named',
                                        layout=widgets.Layout(width='300px'))
named_cam_dd = widgets.Dropdown(options=ALL_CAMERAS, value=vis_state['camera']['named'],
                                description='Camera:', layout=widgets.Layout(width='280px'))

# ── Free-camera sub-controls ──────────────────────────────────────────────────
_FREE_TYPE_OPTIONS = ['free', 'fixed', 'track', 'trackcom']
free_type_dd = widgets.Dropdown(
    options=_FREE_TYPE_OPTIONS, value=vis_state['camera'].get('free_type', 'free'),
    description='Cam type:', layout=widgets.Layout(width='320px'),
    style={'description_width': '100px'},
)
trackbody_txt = widgets.Text(
    value=vis_state['camera'].get('trackbody', ''),
    description='Track body:', placeholder='body name, e.g. thorax',
    layout=widgets.Layout(width='380px'), style={'description_width': '100px'},
)
fixedcam_txt = widgets.Text(
    value=vis_state['camera'].get('fixedcamid', ''),
    description='Fixed cam:', placeholder='named camera, e.g. track1',
    layout=widgets.Layout(width='380px'), style={'description_width': '100px'},
)

def _update_free_subwidgets():
    """Show/hide trackbody / fixedcam inputs based on the selected free_type."""
    ft = free_type_dd.value
    needs_body     = ft in ('track', 'trackcom', 'targetbody', 'targetbodycom')
    needs_fixedcam = ft == 'fixed'
    trackbody_txt.layout.display = '' if needs_body     else 'none'
    fixedcam_txt.layout.display  = '' if needs_fixedcam else 'none'

_free_slider_cfg = [
    ('azimuth',   'Azimuth',   0.0,   0.0,   360.0, 1.0,   '.0f'),
    ('elevation', 'Elevation', -30.0, -90.0, 0.0,   1.0,   '.0f'),
    ('distance',  'Distance',  0.3,   0.02,  2.0,   0.01,  '.3f'),
    ('lookat_x',  'Lookat X',  0.0,  -0.1,   0.1,   0.001, '.3f'),
    ('lookat_y',  'Lookat Y',  0.0,  -0.1,   0.1,   0.001, '.3f'),
    ('lookat_z',  'Lookat Z',  0.0,  -0.1,   0.1,   0.001, '.3f'),
]
free_cam_sliders = {}
for key, desc, val, mn, mx, step, fmt in _free_slider_cfg:
    free_cam_sliders[key] = widgets.FloatSlider(
        value=val, min=mn, max=mx, step=step, description=desc,
        continuous_update=False, readout_format=fmt,
        layout=widgets.Layout(width='420px'), style={'description_width': '100px'},
    )

def _on_cam_mode(change):
    if change['new'] == 'Named':
        vis_state['camera']['mode'] = 'named'
        named_panel.layout.display = ''; free_panel.layout.display = 'none'
    else:
        vis_state['camera']['mode'] = 'free'
        named_panel.layout.display = 'none'; free_panel.layout.display = ''
    do_render()

def _on_named_cam(change): vis_state['camera']['named']     = change['new']; do_render()
def _on_free_az(change):   vis_state['camera']['azimuth']   = change['new']; do_render()
def _on_free_el(change):   vis_state['camera']['elevation']  = change['new']; do_render()
def _on_free_dist(change): vis_state['camera']['distance']  = change['new']; do_render()
def _on_lookat_x(change):  vis_state['camera']['lookat'][0] = change['new']; do_render()
def _on_lookat_y(change):  vis_state['camera']['lookat'][1] = change['new']; do_render()
def _on_lookat_z(change):  vis_state['camera']['lookat'][2] = change['new']; do_render()

def _on_free_type(change):
    vis_state['camera']['free_type'] = change['new']
    _update_free_subwidgets()
    do_render()

def _on_trackbody(change):
    vis_state['camera']['trackbody'] = change['new']
    do_render()

def _on_fixedcam(change):
    vis_state['camera']['fixedcamid'] = change['new']
    do_render()

cam_mode_toggle.observe(_on_cam_mode,   names='value')
named_cam_dd.observe(_on_named_cam,     names='value')
free_cam_sliders['azimuth'].observe(_on_free_az,    names='value')
free_cam_sliders['elevation'].observe(_on_free_el,  names='value')
free_cam_sliders['distance'].observe(_on_free_dist, names='value')
free_cam_sliders['lookat_x'].observe(_on_lookat_x,  names='value')
free_cam_sliders['lookat_y'].observe(_on_lookat_y,  names='value')
free_cam_sliders['lookat_z'].observe(_on_lookat_z,  names='value')
free_type_dd.observe(_on_free_type,     names='value')
trackbody_txt.observe(_on_trackbody,    names='value')
fixedcam_txt.observe(_on_fixedcam,      names='value')

named_panel = widgets.VBox([named_cam_dd])
free_panel  = widgets.VBox([
    free_type_dd,
    trackbody_txt,
    fixedcam_txt,
    *free_cam_sliders.values(),
])
free_panel.layout.display = 'none'
_update_free_subwidgets()   # set initial visibility of trackbody / fixedcam inputs

reset_camera_btn = widgets.Button(description='Reset camera', button_style='warning')
def _on_reset_camera(b):
    global _suppress_render
    _suppress_render = True
    cam = _init_camera_snapshot
    vis_state['camera'] = copy.deepcopy(cam)
    cam_mode_toggle.value = 'Named' if cam['mode'] == 'named' else 'Free orbit'
    named_cam_dd.value    = cam['named']
    free_type_dd.value    = cam.get('free_type', 'free')
    trackbody_txt.value   = cam.get('trackbody', '')
    fixedcam_txt.value    = cam.get('fixedcamid', '')
    for key, sl in free_cam_sliders.items():
        if key == 'lookat_x':   sl.value = cam['lookat'][0]
        elif key == 'lookat_y': sl.value = cam['lookat'][1]
        elif key == 'lookat_z': sl.value = cam['lookat'][2]
        else:                   sl.value = cam.get(key, sl.value)
    _update_free_subwidgets()
    _suppress_render = False
    do_render()
reset_camera_btn.on_click(_on_reset_camera)

tab_camera = widgets.VBox([widgets.HBox([reset_camera_btn]), cam_mode_toggle, named_panel, free_panel])

# ════════════════════════════════════════════════════════════════════════════════
# Tab 5 — Floor
# ════════════════════════════════════════════════════════════════════════════════
floor_color_picker = widgets.ColorPicker(value=vis_state['floor']['color'],
    description='Floor color', concise=False,
    layout=widgets.Layout(width='310px'), style={'description_width': '110px'})
floor_alpha_sl  = widgets.FloatSlider(value=vis_state['floor']['alpha'],  min=0.0, max=1.0,  step=0.05, description='Alpha',        continuous_update=False, readout_format='.2f', layout=widgets.Layout(width='420px'), style={'description_width': '110px'})
floor_texrep_x  = widgets.FloatSlider(value=vis_state['floor']['texrepeat_x'], min=0.5, max=50.0, step=0.5,  description='Grid X',   continuous_update=False, readout_format='.1f', layout=widgets.Layout(width='420px'), style={'description_width': '110px'})
floor_texrep_y  = widgets.FloatSlider(value=vis_state['floor']['texrepeat_y'], min=0.5, max=50.0, step=0.5,  description='Grid Y',   continuous_update=False, readout_format='.1f', layout=widgets.Layout(width='420px'), style={'description_width': '110px'})
floor_refl_sl   = widgets.FloatSlider(value=vis_state['floor']['reflectance'], min=0.0, max=1.0,  step=0.01, description='Reflectance', continuous_update=False, readout_format='.2f', layout=widgets.Layout(width='420px'), style={'description_width': '110px'})
floor_shine_sl  = widgets.FloatSlider(value=vis_state['floor']['shininess'],   min=0.0, max=1.0,  step=0.01, description='Shininess',  continuous_update=False, readout_format='.2f', layout=widgets.Layout(width='420px'), style={'description_width': '110px'})
floor_emis_sl   = widgets.FloatSlider(value=vis_state['floor']['emission'],    min=0.0, max=1.0,  step=0.01, description='Emission',   continuous_update=False, readout_format='.2f', layout=widgets.Layout(width='420px'), style={'description_width': '110px'})

def _on_floor_color(change): vis_state['floor']['color']       = change['new']; do_render()
def _on_floor_alpha(change): vis_state['floor']['alpha']       = change['new']; do_render()
def _on_floor_trx(change):   vis_state['floor']['texrepeat_x'] = change['new']; do_render()
def _on_floor_try(change):   vis_state['floor']['texrepeat_y'] = change['new']; do_render()
def _on_floor_refl(change):  vis_state['floor']['reflectance'] = change['new']; do_render()
def _on_floor_shine(change): vis_state['floor']['shininess']   = change['new']; do_render()
def _on_floor_emis(change):  vis_state['floor']['emission']    = change['new']; do_render()

floor_color_picker.observe(_on_floor_color, names='value')
floor_alpha_sl.observe(_on_floor_alpha,   names='value')
floor_texrep_x.observe(_on_floor_trx,     names='value')
floor_texrep_y.observe(_on_floor_try,     names='value')
floor_refl_sl.observe(_on_floor_refl,     names='value')
floor_shine_sl.observe(_on_floor_shine,   names='value')
floor_emis_sl.observe(_on_floor_emis,     names='value')

# ── Sky / Background ──────────────────────────────────────────────────────────
sky_show_cb = widgets.Checkbox(
    value=vis_state['skybox']['show'], description='Show skybox texture',
    layout=widgets.Layout(width='240px'))
sky_top_picker = widgets.ColorPicker(
    value=vis_state['skybox']['sky_top'], description='Sky top', concise=False,
    layout=widgets.Layout(width='310px'), style={'description_width': '110px'})
sky_bot_picker = widgets.ColorPicker(
    value=vis_state['skybox']['sky_bot'], description='Horizon', concise=False,
    layout=widgets.Layout(width='310px'), style={'description_width': '110px'})

def _on_sky_top(change):   vis_state['skybox']['sky_top'] = change['new']; do_render()
def _on_sky_bot(change):   vis_state['skybox']['sky_bot'] = change['new']; do_render()

sky_top_picker.observe(_on_sky_top, names='value')
sky_bot_picker.observe(_on_sky_bot, names='value')

reset_floor_btn = widgets.Button(description='Reset floor & sky', button_style='warning')
def _on_reset_floor(b):
    global _suppress_render
    _suppress_render = True
    fld = _init_floor_snapshot
    floor_color_picker.value = fld['color']
    floor_alpha_sl.value     = fld['alpha']
    floor_texrep_x.value     = fld['texrepeat_x']
    floor_texrep_y.value     = fld['texrepeat_y']
    floor_refl_sl.value      = fld['reflectance']
    floor_shine_sl.value     = fld['shininess']
    floor_emis_sl.value      = fld['emission']
    sky = _init_sky_snapshot
    vis_state['skybox'] = copy.deepcopy(sky)
    sky_show_cb.value      = sky['show']
    sky_top_picker.value = sky['sky_top']
    sky_bot_picker.value = sky['sky_bot']
    _suppress_render = False
    do_render()
reset_floor_btn.on_click(_on_reset_floor)

if floor_geom_id is not None:
    _mat_note = f', material ID {floor_mat_id}' if floor_mat_id is not None else ' (no material)'
    _floor_lbl = widgets.HTML(f'<b>Floor geom ID {floor_geom_id}{_mat_note}</b>')
else:
    _floor_lbl = widgets.HTML('<b style="color:red">No floor geom found in model</b>')

tab_floor = widgets.VBox([
    _floor_lbl, widgets.HBox([reset_floor_btn]),
    widgets.HTML('<b>Color</b>'), floor_color_picker, floor_alpha_sl,
    widgets.HTML('<br><b>Grid / Texture spacing</b>'), floor_texrep_x, floor_texrep_y,
    widgets.HTML('<br><b>Material properties</b>'), floor_refl_sl, floor_shine_sl, floor_emis_sl,
    widgets.HTML('<hr><b>Sky / Background</b>'),
    sky_show_cb, sky_top_picker, sky_bot_picker,
])
# ════════════════════════════════════════════════════════════════════════════════
# Tab 6 — Settings (Save / Load + Camera Presets)
# ════════════════════════════════════════════════════════════════════════════════

def _apply_cam_widgets(cam):
    """Push a camera dict into the camera tab widgets (no render)."""
    cam_mode_toggle.value = 'Named' if cam['mode'] == 'named' else 'Free orbit'
    named_cam_dd.value    = cam['named']
    free_type_dd.value    = cam.get('free_type', 'free')
    trackbody_txt.value   = cam.get('trackbody', '')
    fixedcam_txt.value    = cam.get('fixedcamid', '')
    for key, sl in free_cam_sliders.items():
        if key == 'lookat_x':   sl.value = cam['lookat'][0]
        elif key == 'lookat_y': sl.value = cam['lookat'][1]
        elif key == 'lookat_z': sl.value = cam['lookat'][2]
        else:                   sl.value = cam.get(key, sl.value)
    _update_free_subwidgets()

def _apply_settings_to_widgets():
    """Sync all widgets from vis_state after a JSON load."""
    global _suppress_render
    _suppress_render = True
    for cat in BODY_CATEGORIES:
        color_pickers[cat].value = vis_state['colors'][cat]
        # Reset per-geom dropdowns to "All" since per-geom colors may have changed
        geom_dd[cat].value = _ALL_LABEL
        geom_picker[cat].unobserve_all()
        geom_picker[cat].value = vis_state['colors'][cat]
        geom_picker[cat].observe(_geom_picker_cb[cat], names='value')
    alpha_slider.value = vis_state['alpha']
    for key in vis_state['vis_flags']:
        if key in vis_checkboxes:
            vis_checkboxes[key].value = vis_state['vis_flags'][key]
    for g in range(6):
        geom_group_btns[g].value = vis_state['geom_groups'][g]
        geom_group_btns[g].button_style = 'info' if vis_state['geom_groups'][g] else ''
        site_group_btns[g].value = vis_state['site_groups'][g]
        site_group_btns[g].button_style = 'info' if vis_state['site_groups'][g] else ''
    for li, ld in enumerate(vis_state['lighting']['lights']):
        lw = lighting_widgets.get(li)
        if lw is None: continue
        lw['active'].value = ld['active']
        for ch in range(3):
            lw['ambient'][ch].value  = ld['ambient'][ch]
            lw['diffuse'][ch].value  = ld['diffuse'][ch]
            lw['specular'][ch].value = ld['specular'][ch]
        lw['dir_az'].value = ld['dir_az']
        lw['dir_el'].value = ld['dir_el']
    hl = vis_state['lighting']['headlight']
    hw = lighting_widgets['headlight']
    hw['active'].value = hl['active']
    for ch in range(3):
        hw['ambient'][ch].value  = hl['ambient'][ch]
        hw['diffuse'][ch].value  = hl['diffuse'][ch]
        hw['specular'][ch].value = hl['specular'][ch]
    # ── Lighting presets (new keys; ensure present even for old JSON) ────
    vis_state['lighting'].setdefault('use_dual_lighting',   False)
    vis_state['lighting'].setdefault('use_scale_lights',    False)
    vis_state['lighting'].setdefault('scale_lights_factor', 1.25)
    dual_light_cb.value  = vis_state['lighting']['use_dual_lighting']
    scale_light_cb.value = vis_state['lighting']['use_scale_lights']
    scale_sl.value       = vis_state['lighting']['scale_lights_factor']
    scale_sl.layout.display = '' if vis_state['lighting']['use_scale_lights'] else 'none'
    _apply_cam_widgets(vis_state['camera'])
    floor_color_picker.value = vis_state['floor']['color']
    floor_alpha_sl.value     = vis_state['floor']['alpha']
    floor_texrep_x.value     = vis_state['floor']['texrepeat_x']
    floor_texrep_y.value     = vis_state['floor']['texrepeat_y']
    floor_refl_sl.value      = vis_state['floor']['reflectance']
    floor_shine_sl.value     = vis_state['floor']['shininess']
    floor_emis_sl.value      = vis_state['floor']['emission']
    sky = vis_state.get('skybox', {})
    if sky:
        sky_show_cb.value      = sky.get('show', True)
        sky_top_picker.value = sky.get('sky_top', _init_sky_top_hex)
        sky_bot_picker.value = sky.get('sky_bot', _init_sky_bot_hex)
    _suppress_render = False
    do_render()

# ── Save / Load ───────────────────────────────────────────────────────────────
save_path_text = widgets.Text(value='pose_tuner_settings.json', description='File:',
                               layout=widgets.Layout(width='420px'),
                               style={'description_width': '50px'})
save_btn = widgets.Button(description='Save settings', button_style='success',
                           layout=widgets.Layout(width='140px'))
load_btn = widgets.Button(description='Load settings', button_style='info',
                           layout=widgets.Layout(width='140px'))
settings_status = widgets.HTML('')

def _on_save(b):
    try:
        save_settings(save_path_text.value)
        settings_status.value = f'<span style="color:green">✓ Saved → {save_path_text.value}</span>'
    except Exception as e:
        settings_status.value = f'<span style="color:red">Save error: {e}</span>'

def _on_load(b):
    global _suppress_render
    # Suppress all renders triggered by widget value changes while loading the
    # preset data and rebuilding the dropdown.  _apply_settings_to_widgets()
    # will set _suppress_render = False and do a single clean render at the end.
    _suppress_render = True
    try:
        with open(save_path_text.value, 'r') as _f:
            _data = json.load(_f)
        if 'camera_presets' in _data:
            saved_camera_presets.update(_data.pop('camera_presets'))
            _refresh_presets_dd()
        # Restore geom_colors with int keys (JSON stores them as strings)
        if 'geom_colors' in _data:
            vis_state['geom_colors'] = {int(k): v for k, v in _data.pop('geom_colors').items()}
        # geom_render_state is read-only metadata — skip restoring into vis_state
        _data.pop('geom_render_state', None)
        for k, v in _data.items():
            if k in vis_state:
                vis_state[k] = v
        _apply_settings_to_widgets()  # handles _suppress_render = False; do_render()
        settings_status.value = f'<span style="color:green">✓ Loaded ← {save_path_text.value}</span>'
    except FileNotFoundError:
        _suppress_render = False
        settings_status.value = f'<span style="color:red">File not found: {save_path_text.value}</span>'
    except Exception as e:
        _suppress_render = False
        settings_status.value = f'<span style="color:red">Load error: {e}</span>'

save_btn.on_click(_on_save)
load_btn.on_click(_on_load)

# ── Camera Presets ────────────────────────────────────────────────────────────
preset_name_txt = widgets.Text(value='my_view', description='Name:',
                                layout=widgets.Layout(width='260px'),
                                style={'description_width': '50px'})
save_preset_btn   = widgets.Button(description='Save camera',   button_style='success', layout=widgets.Layout(width='125px'))
preset_dd = widgets.Dropdown(options=['(none)'], description='Preset:',
                              layout=widgets.Layout(width='260px'),
                              style={'description_width': '60px'})
load_preset_btn   = widgets.Button(description='Load camera',   button_style='info',   layout=widgets.Layout(width='125px'))
delete_preset_btn = widgets.Button(description='Delete',        button_style='danger', layout=widgets.Layout(width='80px'))
preset_status = widgets.HTML('')

def _refresh_presets_dd():
    preset_dd.options = ['(none)'] + list(saved_camera_presets.keys())

def _on_save_preset(b):
    name = preset_name_txt.value.strip()
    if not name:
        preset_status.value = '<span style="color:red">Enter a name first</span>'; return
    saved_camera_presets[name] = copy.deepcopy(vis_state['camera'])
    _refresh_presets_dd()
    preset_dd.value = name
    preset_status.value = f'<span style="color:green">✓ Saved camera "{name}"</span>'

def _on_load_preset(b):
    sel = preset_dd.value
    if sel == '(none)' or sel not in saved_camera_presets:
        preset_status.value = '<span style="color:orange">Select a preset first</span>'; return
    global _suppress_render
    _suppress_render = True
    cam = saved_camera_presets[sel]
    vis_state['camera'] = copy.deepcopy(cam)
    _apply_cam_widgets(cam)
    _suppress_render = False
    do_render()
    preset_status.value = f'<span style="color:green">✓ Loaded camera "{sel}"</span>'

def _on_delete_preset(b):
    sel = preset_dd.value
    if sel == '(none)' or sel not in saved_camera_presets:
        preset_status.value = '<span style="color:orange">Select a preset first</span>'; return
    del saved_camera_presets[sel]
    _refresh_presets_dd()
    preset_status.value = f'Deleted "{sel}"'
save_preset_btn.on_click(_on_save_preset)
load_preset_btn.on_click(_on_load_preset)
delete_preset_btn.on_click(_on_delete_preset)

tab_settings = widgets.VBox([
    widgets.HTML('<b>Save / Load All Settings</b>'),
    save_path_text,
    widgets.HBox([save_btn, load_btn]),
    settings_status,
    widgets.HTML('<hr><b>Camera Presets</b>'),
    widgets.HBox([preset_name_txt, save_preset_btn]),
    widgets.HBox([preset_dd, load_preset_btn, delete_preset_btn]),
    preset_status,
])

# ════════════════════════════════════════════════════════════════════════════════
# Assemble & display
# ════════════════════════════════════════════════════════════════════════════════
main_tabs = widgets.Tab(children=[tab_pose, tab_colors, tab_vis, tab_lighting,
                                   tab_camera, tab_floor, tab_settings])
for i, name in enumerate(['Pose', 'Colors', 'Visualization', 'Lighting',
                           'Camera', 'Floor', 'Settings']):
    main_tabs.set_title(i, name)

controls = widgets.VBox([main_tabs], layout=widgets.Layout(width='650px'))
display(widgets.HBox([controls, out]))
do_render()


In [ ]:
def render_video(
    qposes,
    settings,
    height=480,
    width=640,
    fps=30,
    show=True,
    camera=None,
):
    """Render a video from a qpos trajectory using saved pose_tuner visual settings.

    Applies all colors, floor, lighting, skybox, vis flags, and camera from a
    saved pose_tuner JSON (or a pre-loaded dict) and renders one frame per qpos.

    Args:
        qposes:   (T, nq) array of joint positions (one row per timestep).
        settings: str path to a pose_tuner JSON file, or the already-loaded dict.
        height:   output frame height in pixels (default 480).
        width:    output frame width in pixels (default 640).
        fps:      playback frame rate for display (default 30).
        show:     if True, display the video inline with mediapy.
        camera:   override the camera from the settings JSON.
                  - None (default): use whatever camera is stored in the JSON.
                  - str: named camera, e.g. 'track1', 'side', 'back', 'bottom'.
                    Available names: ALL_CAMERAS.
                  - mujoco.MjvCamera: a free-orbit camera built manually.

    Returns:
        np.ndarray of shape (T, height, width, 3), dtype uint8.
    """
    if isinstance(settings, str):
        with open(settings) as _f:
            settings = json.load(_f)

    # Apply geom colors, floor, lighting, skybox -- modifies mj_model in-place.
    _settings_camera = apply_render_settings(mj_model, settings)

    # Use caller-supplied camera if given, otherwise fall back to settings camera.
    camera = camera if camera is not None else _settings_camera

    # Build MjvOption from saved vis flags / geom+site group toggles.
    vf  = settings.get("vis_flags", {})
    gg  = settings.get("geom_groups",  [1, 1, 1, 1, 1, 1])
    sg  = settings.get("site_groups",  [1, 1, 1, 1, 1, 1])
    opt = mujoco.MjvOption()
    opt.flags[mujoco.mjtVisFlag.mjVIS_CONTACTPOINT] = vf.get("contact_points", False)
    opt.flags[mujoco.mjtVisFlag.mjVIS_CONTACTFORCE] = vf.get("contact_forces",  False)
    opt.flags[mujoco.mjtVisFlag.mjVIS_ACTUATOR]     = vf.get("actuators",       False)
    opt.flags[mujoco.mjtVisFlag.mjVIS_JOINT]        = vf.get("joints",          False)
    opt.flags[mujoco.mjtVisFlag.mjVIS_TRANSPARENT]  = vf.get("transparent",     False)
    for k in range(min(6, len(opt.geomgroup))):
        opt.geomgroup[k] = int(gg[k])
    for k in range(min(6, len(opt.sitegroup))):
        opt.sitegroup[k] = int(sg[k])

    show_shadows   = vf.get("shadows",   True)
    show_wireframe = vf.get("wireframe", False)
    show_skybox    = settings.get("skybox", {}).get("show", True)

    frames = []
    with mujoco.Renderer(mj_model, height=height, width=width) as renderer:
        for qpos in qposes:
            mj_data.qpos[:] = qpos
            mujoco.mj_forward(mj_model, mj_data)
            renderer.update_scene(mj_data, camera=camera, scene_option=opt)
            if not show_shadows:
                renderer.scene.flags[mujoco.mjtRndFlag.mjRND_SHADOW]   = False
            if show_wireframe:
                renderer.scene.flags[mujoco.mjtRndFlag.mjRND_WIREFRAME] = True
            if not show_skybox:
                renderer.scene.flags[mujoco.mjtRndFlag.mjRND_SKYBOX]   = False
            for _fn, _kw in _build_scene_modifiers():
                _fn(renderer.scene, mj_data, **_kw)
            frames.append(renderer.render().copy())

    video = np.stack(frames)
    if show:
        media.show_video(video, fps=fps)
    return video


# ── Usage example ──────────────────────────────────────────────────────────────────────────────
# Replace qposes_example with your (T, nq) trajectory array.
# qposes_example = np.tile(mj_model.qpos_spring, (60, 1))

# Use the camera saved in the JSON (default):
# video = render_video(
#     qposes_example,
#     '/home/eabe/Research/MyRepos/fly_neuromech/notebooks/Purple.json',
#     height=480, width=640, fps=30,
# )

# Override with a named camera:
# video = render_video(
#     qposes_example,
#     '/home/eabe/Research/MyRepos/fly_neuromech/notebooks/Purple.json',
#     camera='side',   # any name from ALL_CAMERAS
# )

# Override with a free-orbit camera:
# free_cam = mujoco.MjvCamera()
# free_cam.type      = mujoco.mjtCamera.mjCAMERA_FREE
# free_cam.azimuth   = 45.0
# free_cam.elevation = -20.0
# free_cam.distance  = 0.4
# free_cam.lookat[:] = [0.0, 0.0, 0.0]
# video = render_video(qposes_example, 'Purple.json', camera=free_cam)


# Camera Pans

In [ ]:

# ── Camera-pan utilities ───────────────────────────────────────────────────────

def _lerp_angle(a, b, t):
    """Interpolate from angle *a* to *b* along the shortest arc (handles 360→0 wrap)."""
    diff = ((b - a + 180.0) % 360.0) - 180.0
    return a + diff * t


def _cosine_ease(t):
    """Cosine easing: starts and ends slow, fastest in the middle.  t in [0, 1]."""
    return 0.5 * (1.0 - np.cos(np.pi * t))


def _resolve_preset(cam_cfg):
    """Return full camera parameter dict from a camera_presets entry.

    pose_tuner stores azimuth/elevation/distance/lookat in every preset
    (even when mode='named'), so we read them directly.  Also reads
    free_type, trackbody, and fixedcamid so that tracking/target cameras
    are reproduced correctly during a pan.
    """
    return {
        "azimuth":    float(cam_cfg.get("azimuth",    180.0)),
        "elevation":  float(cam_cfg.get("elevation",  -20.0)),
        "distance":   float(cam_cfg.get("distance",    0.5)),
        "lookat":     [float(v) for v in cam_cfg.get("lookat", [0.0, 0.0, 0.0])],
        "free_type":  cam_cfg.get("free_type",  "free"),
        "trackbody":  cam_cfg.get("trackbody",  ""),
        "fixedcamid": cam_cfg.get("fixedcamid", ""),
    }


# Backward-compat alias
_resolve_to_free = _resolve_preset


def _build_pan_camera(A, B, t):
    """Interpolate between two resolved keyframe dicts at position t in [0, 1].

    Camera type is snapped at the midpoint when the two keyframes use different
    types (t < 0.5 → A's type; t >= 0.5 → B's type).  azimuth/elevation/
    distance/lookat are always smoothly interpolated regardless of type,
    because they carry meaningful offset information for track/targetbody modes
    as well as for free-orbit.

    Args:
        A, B: dicts from _resolve_preset().
        t:    interpolation position in [0, 1].

    Returns:
        mujoco.MjvCamera
    """
    kf = A if t < 0.5 else B
    free_type = kf["free_type"]
    mj_type, needs_body, needs_fixedcam = _FREE_TYPE_MAP.get(free_type, _FREE_TYPE_MAP["free"])

    cam = mujoco.MjvCamera()
    cam.type      = mj_type
    cam.azimuth   = _lerp_angle(A["azimuth"],   B["azimuth"],   t)
    cam.elevation = A["elevation"] + (B["elevation"] - A["elevation"]) * t
    cam.distance  = A["distance"]  + (B["distance"]  - A["distance"])  * t
    cam.lookat[:] = [
        A["lookat"][i] + (B["lookat"][i] - A["lookat"][i]) * t
        for i in range(3)
    ]

    if needs_fixedcam:
        cam_id = mujoco.mj_name2id(mj_model, mujoco.mjtObj.mjOBJ_CAMERA, kf.get("fixedcamid", ""))
        cam.fixedcamid = max(cam_id, 0)

    if needs_body:
        body_id = mujoco.mj_name2id(mj_model, mujoco.mjtObj.mjOBJ_BODY, kf.get("trackbody", ""))
        cam.trackbodyid = max(body_id, 0)

    return cam


def make_pan_cameras(settings, preset_names, total_frames=120,
                     segment_weights=None, loop=False):
    """Build a list of MjvCamera objects that smoothly pan between saved presets.

    Each segment (transition between two consecutive keyframes) is interpolated
    with cosine easing.  segment_weights lets you allocate more frames — and
    therefore more screen time — to specific transitions.

    Camera types (free_type) are fully supported.  For free-orbit, track,
    trackcom, targetbody, and targetbodycom presets the azimuth/elevation/
    distance/lookat values are smoothly interpolated.  For fixed presets the
    camera snaps to the named MuJoCo camera at each frame.  When two keyframes
    have different types the type snaps at the segment midpoint.

    Args:
        settings:         str path to a pose_tuner JSON, or already-loaded dict.
        preset_names:     ordered list of preset name strings (keys in camera_presets).
                          At least two names are required.
        total_frames:     total number of camera frames to generate (default 120).
        segment_weights:  relative time budget for each segment, as a list of
                          positive numbers with length == n_segments.
                          Values are normalised automatically, so [3, 1] and
                          [0.75, 0.25] are equivalent.
                          None (default) gives every segment equal time.
        loop:             if True, append a final segment from the last preset back
                          to the first for a seamless loop.

    Returns:
        list of mujoco.MjvCamera.  Length is total_frames (may differ slightly
        from the requested value due to integer rounding; the last segment absorbs
        the rounding remainder).

    Examples:
        # Equal time on both segments (default)
        cameras = make_pan_cameras('Purple.json',
                                   ['side_close1', 'side_close2'],
                                   total_frames=120)

        # Spend 3x as long on the first segment as the second
        cameras = make_pan_cameras('Purple.json',
                                   ['side_close1', 'side_close2', 'front'],
                                   total_frames=180,
                                   segment_weights=[3, 1])

        # Loop through three presets, dwelling longest at the first
        cameras = make_pan_cameras('Purple.json',
                                   ['side_close1', 'side_close2', 'front'],
                                   total_frames=240,
                                   segment_weights=[0.5, 0.25, 0.25],
                                   loop=True)
    """
    if isinstance(settings, str):
        with open(settings) as _f:
            settings = json.load(_f)

    presets = settings.get("camera_presets", {})
    if len(preset_names) < 2:
        raise ValueError("Need at least two preset names to interpolate between.")

    keyframes = []
    for name in preset_names:
        if name not in presets:
            raise KeyError(
                f"Preset '{name}' not found. Available: {list(presets.keys())}"
            )
        keyframes.append(_resolve_preset(presets[name]))

    if loop:
        keyframes.append(keyframes[0])  # close the loop

    n_segs = len(keyframes) - 1

    # --- resolve per-segment frame counts from weights -------------------------
    if segment_weights is None:
        weights = [1.0] * n_segs
    else:
        if len(segment_weights) != n_segs:
            raise ValueError(
                f"segment_weights has {len(segment_weights)} entries but there are "
                f"{n_segs} segments (preset_names has {len(preset_names)} entries"
                + (", plus the loop-back segment" if loop else "") + ")."
            )
        weights = [float(w) for w in segment_weights]

    total_w = sum(weights)
    # Allocate frames proportionally; last segment absorbs rounding remainder.
    seg_frames = [max(1, round(w / total_w * total_frames)) for w in weights]
    seg_frames[-1] = max(1, total_frames - sum(seg_frames[:-1]))

    # --- build per-frame MjvCamera list ---------------------------------------
    cameras = []
    for seg in range(n_segs):
        A = keyframes[seg]
        B = keyframes[seg + 1]
        n = seg_frames[seg]
        for fi in range(n):
            t = _cosine_ease(fi / n)
            cameras.append(_build_pan_camera(A, B, t))

    return cameras


# ── Muscle tendon coloring helpers ─────────────────────────────────────────────

def _build_tendon_color_map(model):
    """Pre-compute per-actuator base RGBA from functional group colors.

    Returns:
        act_to_ten: dict mapping actuator index → tendon index
        base_rgba:  (nu, 4) float32 array of functional-group base colors
    """
    from fly_neuromechanics.utils.plot_utils import actuator_color
    import matplotlib.colors as mcolors

    act_to_ten = {}
    base_rgba = np.zeros((model.nu, 4), dtype=np.float32)
    _mjTRN_TENDON = int(mujoco.mjtTrn.mjTRN_TENDON)

    for i in range(model.nu):
        name = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_ACTUATOR, i)
        if name is None:
            continue
        trnid = model.actuator_trnid[i, 0]
        trntype = int(model.actuator_trntype[i])
        if trntype == _mjTRN_TENDON and 0 <= trnid < model.ntendon:
            act_to_ten[i] = trnid
            hex_clr = actuator_color(name, by='function')
            r, g, b = mcolors.hex2color(hex_clr)
            base_rgba[i] = [r, g, b, 1.0]

    print(f"Mapped {len(act_to_ten)} actuators to tendons (mjTRN_TENDON={_mjTRN_TENDON})")
    return act_to_ten, base_rgba


def _build_scene_modifiers_from_settings(settings):
    """Build scene modifier list from a JSON settings dict (not vis_state)."""
    lighting = settings.get('lighting', {})
    mods = []
    if lighting.get('use_dual_lighting', False):
        mods.append((dual_lighting, {}))
    if lighting.get('use_scale_lights', False):
        mods.append((scale_lights, {'scale': lighting.get('scale_lights_factor', 1.25)}))
    return mods


def render_video_pan(qposes, settings, cameras, height=480, width=640, fps=30,
                     show=True, modify_scene_fns=None, state=None, ctrls=None,
                     tendon_width=0.003, tendon_min_width=0.0005):
    """Render a video where the camera follows a smooth pan trajectory.

    Combines render_video-style visual settings with per-frame cameras from
    make_pan_cameras().

    Args:
        qposes:   (T, nq) array — must have the same length as *cameras*.
        settings: str path to pose_tuner JSON, or already-loaded dict.
        cameras:  list of MjvCamera from make_pan_cameras() (length == T).
        height, width, fps, show: same as render_video().
        ctrls:    optional (T, nu) array of control signals. When provided,
                  tendons are colored by functional muscle group with alpha
                  and width scaled by ctrl magnitude.
        tendon_width:     max tendon rendering width at full activation (default 0.003).
        tendon_min_width: min tendon width at zero activation (default 0.0005).

    Returns:
        np.ndarray (T, height, width, 3) uint8.
    """
    if isinstance(settings, str):
        with open(settings) as _f:
            settings = json.load(_f)

    # Apply colours / lighting / skybox from the saved settings.
    apply_render_settings(mj_model, settings)

    vf  = settings.get("vis_flags", {})
    gg  = settings.get("geom_groups",  [1, 1, 1, 1, 1, 1])
    sg  = settings.get("site_groups",  [1, 1, 1, 1, 1, 1])
    opt = mujoco.MjvOption()
    opt.flags[mujoco.mjtVisFlag.mjVIS_CONTACTPOINT] = vf.get("contact_points", False)
    opt.flags[mujoco.mjtVisFlag.mjVIS_CONTACTFORCE] = vf.get("contact_forces",  False)
    opt.flags[mujoco.mjtVisFlag.mjVIS_ACTUATOR]     = vf.get("actuators",       False)
    opt.flags[mujoco.mjtVisFlag.mjVIS_JOINT]        = vf.get("joints",          False)
    opt.flags[mujoco.mjtVisFlag.mjVIS_TRANSPARENT]  = vf.get("transparent",     False)
    for k in range(min(6, len(opt.geomgroup))):
        opt.geomgroup[k] = int(gg[k])
    for k in range(min(6, len(opt.sitegroup))):
        opt.sitegroup[k] = int(sg[k])

    show_shadows   = vf.get("shadows",   True)
    show_wireframe = vf.get("wireframe", False)
    show_skybox    = settings.get("skybox", {}).get("show", True)

    # ── Scene-level lighting modifiers from settings ───────────────────────
    scene_mods = _build_scene_modifiers_from_settings(settings)

    # ── Muscle visualization setup ─────────────────────────────────────────
    show_muscles = ctrls is not None
    if show_muscles:
        ctrls = np.asarray(ctrls)
        opt.flags[mujoco.mjtVisFlag.mjVIS_TENDON] = True
        act_to_ten, base_rgba = _build_tendon_color_map(mj_model)
        orig_tendon_rgba = mj_model.tendon_rgba.copy()
        orig_tendon_width = mj_model.tendon_width.copy()
        # Hide non-muscle tendons (set alpha=0)
        muscle_ten_ids = set(act_to_ten.values())
        for t in range(mj_model.ntendon):
            if t not in muscle_ten_ids:
                mj_model.tendon_rgba[t, 3] = 0.0
        width_range = tendon_width - tendon_min_width

    frames = []
    with mujoco.Renderer(mj_model, height=height, width=width) as renderer:
        for i, qpos in enumerate(qposes):
            mj_data.qpos[:] = qpos
            if show_muscles:
                mj_data.ctrl[:] = ctrls[i]
            mujoco.mj_forward(mj_model, mj_data)

            # ── Per-frame tendon coloring + width ─────────────────────
            if show_muscles:
                ctrl_i = ctrls[i]
                ctrl_max = max(np.abs(ctrl_i).max(), 1e-8)
                for act_id, ten_id in act_to_ten.items():
                    norm = float(np.clip(np.abs(ctrl_i[act_id]) / ctrl_max, 0.0, 1.0))
                    alpha = max(norm, 0.05)
                    mj_model.tendon_rgba[ten_id] = base_rgba[act_id] * np.array([1, 1, 1, alpha])
                    mj_model.tendon_width[ten_id] = tendon_min_width + width_range * norm

            renderer.update_scene(mj_data, camera=cameras[i], scene_option=opt)
            if not show_shadows:
                renderer.scene.flags[mujoco.mjtRndFlag.mjRND_SHADOW]   = False
            if show_wireframe:
                renderer.scene.flags[mujoco.mjtRndFlag.mjRND_WIREFRAME] = True
            if not show_skybox:
                renderer.scene.flags[mujoco.mjtRndFlag.mjRND_SKYBOX]   = False

            # ── Apply scene-level lighting (dual_lighting, scale_lights) ──
            for _fn, _kw in scene_mods:
                _fn(renderer.scene, state, geom_xpos=mj_data.geom_xpos, **_kw)

            if modify_scene_fns is not None:
                for k in range(len(modify_scene_fns)):
                    is_rollout = isinstance(state, Rollout)
                    is_tuple = isinstance(state, tuple)

                    if is_rollout or is_tuple:
                        rollout_info = state.info if is_rollout else None
                        modify_scene_fns[k](
                            scene=renderer.scene,
                            state=state,
                            rollout_info=rollout_info,
                            timestep_idx=i,
                            geom_xpos=mj_data.geom_xpos
                        )
            frames.append(renderer.render().copy())

    # Restore original tendon rgba and width
    if show_muscles:
        mj_model.tendon_rgba[:] = orig_tendon_rgba
        mj_model.tendon_width[:] = orig_tendon_width

    video = np.stack(frames)
    if show:
        media.show_video(video, fps=fps)
    return video


In [ ]:
from fly_mimic.envs.fruitfly.constants import _WING_PARAMS
from fly_mimic.utils.data_utils import ReferenceClips

clip_idx = 0 # jnp.argmax(ref_clips.clip_lengths)
clip_length = 400 # ref_clips.clip_lengths[clip_idx]

single_clip = saver.load_rollout(clip_idx, clip_length=clip_length)
qposes_pan = single_clip.qpos[:clip_length]
abd_idxs = [get_qpos_ids(mj_model, [joint.name for joint in floor_spec.joints if ('abdomen' in joint.name)])]
wing_joint_idxs = [get_qpos_ids(mj_model, [joint.name for joint in floor_spec.joints if ('wing' in joint.name)])]
if data_type == 'V1_walking':
    JSON_PATH = project_dir / 'notebooks/Earthy_V1_contact.json'
    new_qpos = qposes_pan.at[:, wing_joint_idxs].set(0.0)
    ctrls=None
    camera_pans = ['top_track1', 'tracking_close3', 'tracking_close2', 'tracking_close1', 'tracking_close4', 'tracking_close5']
    segment_weights = [.75, 1, 1, 1, 1, .75]
elif (data_type == 'V2_walking') or (data_type == 'V2_stationary'):
    JSON_PATH = project_dir / 'notebooks/Earthy.json'
    new_qpos = qposes_pan.at[:, wing_joint_idxs].set(_WING_PARAMS['default_qpos'])
    ctrls=single_clip.ctrl
    camera_pans = ['tracking_close3', 'tracking_close2', 'tracking_close1', 'tracking_close4', 'tracking_close5']
    segment_weights = [1, 1, 1, 1, 1]
elif data_type == 'V1_flight':
    JSON_PATH = project_dir / 'notebooks/Earthy_V1_flight.json'
    ctrls=None
    camera_pans = ['top_track1', 'tracking_close3', 'tracking_close2', 'tracking_close1', 'tracking_close4', 'tracking_close5']
    segment_weights = [.75, 1, 1, 1, 1, .75]
qposes_pan = new_qpos.at[:, abd_idxs].set(0.0)

# ── Usage example ──────────────────────────────────────────────────────────────

# Spend 3x longer on the side_close1 → side_close2 segment than the loop-back.
# total_frames=120, weights=[3,1] → seg0 gets 90 frames, seg1 gets 30 frames.
cameras = make_pan_cameras(
    JSON_PATH.as_posix(),
    camera_pans,    
    total_frames=clip_length,
    segment_weights=segment_weights,   # relative dwell; omit (or set None) for equal time
    loop=True,
)
print(f"Generated {len(cameras)} camera frames")
height=2*512
width=2*512
# video_pan = render_video_pan(qposes_pan, JSON_PATH, cameras, height=480, width=640, fps=50, ctrls=single_clip.ctrl)
video_pan = render_video_pan(qposes_pan, JSON_PATH.as_posix(), cameras, height=height, width=width, fps=50, ctrls=ctrls)


In [ ]:

tracees = media.read_video('/gscratch/portia/eabe/fly_neuromech/walk/33833601/figures/Traces.mp4')

In [ ]:
video_pan.shape,tracees.shape

In [ ]:
concat_vid = np.concatenate([video_pan[1:], tracees], axis=2)
media.show_video(concat_vid, fps=40, title='combined')

In [ ]:
video = render_video(
    qposes_pan,
    JSON_PATH.as_posix(),
    height=480,
    width=640,
    fps=60,
    camera='track1'
)

In [ ]:
ref_clips.clip_lengths.shape

In [ ]:
from fly_neuromechanics.vnc_utils import load_wTable
# wTable = load_wTable('/gscratch/portia/eabe/fly_neuromech/data/BANC/wTable_20260226_all_neurons_processed.csv')
all_mn_idx  = jnp.hstack([jnp.asarray(grp.neuron_indices) for grp in env.mu_mapper.groups.values()])
all_sn_idx  = jnp.hstack([jnp.asarray(grp.neuron_indices) for grp in env.sen_mapper.groups.values()])

In [ ]:

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
max_frames = 300
H = single_clip.info['vnc_rates']
H_sensory = H[:max_frames, all_sn_idx]  # (max_frames, n_sensory)
H_motor   = H[:max_frames, all_mn_idx]
# H_inter   = H[:max_frames, interneuron_indices]
print(f"Using H from saver, ignoring clip_idx since it's already sliced. H.shape={H.shape}")
# # print(f"H_sensory.shape={H_sensory.shape}, H_motor.shape={H_motor.shape}, H_inter.shape={H_inter.shape}")
def pick_top_active(arr, k, normalize=True):
    std = arr.std(axis=0)
    idx = np.sort(np.argsort(std)[-k:])
    selected = arr[:, idx]
    if normalize:
        rms = np.sqrt(np.mean(arr**2, axis=0))
        idx = np.sort(np.argsort(rms)[-k:])
        selected = arr[:, idx]
        mu = selected.mean(axis=0, keepdims=True)
        sigma = selected.std(axis=0, keepdims=True)
        normalized = (selected - mu) / np.where(sigma == 0, 1, sigma)
    else:
        normalized = selected
    return normalized
max_per_panel = 2
norm = True
S = pick_top_active(H_sensory, max_per_panel, normalize=norm)
M = pick_top_active(H_motor,   max_per_panel, normalize=norm)
I = pick_top_active(H_inter,   max_per_panel, normalize=norm)
# 2 neurons per type, each on its own row; green=sensory, red=motor, blue=inter
neuron_rows = [
    (S[:, 0], "#2e7d32"),  # sensory 1 - dark green
    (S[:, 1], "#66bb6a"),  # sensory 2 - light green
    (M[:, 0], "#b71c1c"),  # motor 1   - dark red
    (M[:, 1], "#ef5350"),  # motor 2   - light red
    (I[:, 0], "#1565c0"),  # inter 1   - dark blue
    (I[:, 1], "#42a5f5"),  # inter 2   - light blue
]
def render_frame(t):
    fig, axs = plt.subplots(6, 1, figsize=(14, 10), sharex=True, constrained_layout=True)
    fig.patch.set_facecolor("#0d0d0d")
    xs = np.arange(t + 1)
    for ax, (trace, color) in zip(axs, neuron_rows):
        ax.set_facecolor("#0d0d0d")
        ax.set_xlim(0, max_frames)
        ax.set_ylim(-3, 3)
        ax.set_yticks([])
        ax.set_xticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)
        ax.plot(xs, trace[:t+1], lw=2.5, alpha=0.9, color=color)
        ax.axvline(x=t, color='white', lw=1, alpha=0.5, linestyle='--')
    fig.canvas.draw()
    buf = fig.canvas.buffer_rgba()
    frame = np.frombuffer(buf, dtype=np.uint8).reshape(
        fig.canvas.get_width_height()[::-1] + (4,)
    )[..., :3].copy()
    plt.close(fig)
    return t, frame
print(f"Rendering {max_frames} frames...")
results = [render_frame(t) for t in range(max_frames)]
results.sort(key=lambda x: x[0])
neural_rendered_frames = [f for _, f in results]
print("Done.")
media.show_video(neural_rendered_frames, fps=50)
